# 06 — ACDC / MRzero : 32 échos, bruit indépendant et relaxation native

Ce notebook reprend le pipeline du **05** avec la cartographie
`megre_b0_s0_t2star_128x128_fov200mm_32echo.seq` : **32 échos monopolaire 128×128,
TE de 2,123 à 58,543 ms**, FOV 200 mm et coupe 5 mm. La GRE-EPI corrigée reste la
séquence EPI 128×128 du 05 ; le nouveau fichier remplace la séquence de cartographie B0/relaxation.

Le bruit complexe des récepteurs est estimé dans des coins définis indépendamment du
masque objet. La covariance sert à débiaiser le RSS, puis le support combine SNR,
seuil relatif et composantes spatiales. Les cartes B0 utilisent les premiers échos
pour limiter l'effet des échos tardifs faibles ; la relaxation exploite les 32 échos.
Le mode historique conserve ses critères de fiabilité ; le nouveau mode complexe conserve les pertes même lorsque sa qualité est insuffisante, en les signalant.

**Acquisition nécessaire :** renseigner `B0_SCAN_PATH` avec le `.dat` mesuré avec
cette séquence à 32 échos. `None` recherche un seul `*b0_map.dat` dans `INPUT_DIR`,
puis vérifie sa compatibilité. Le `.dat` historique à trois échos est refusé :
changer de `.seq` ne crée pas les échos manquants.

Le contrôle initial affiche les TE réellement calculés et l'extrapolation vers le TE
et les derniers ADC EPI. À TE ≈78 ms, il reste environ 19,5 ms après le dernier écho ;
les derniers ADC sont plus tardifs. Ces échos ne mesurent ni T1 ni le B0 suivant z.

Shim statique, optimisation dynamique, orientation, réponse matérielle et export CSV
reprennent le 05. Les résultats ont un dossier `results_experiment_echo32/mrzero06_*`
propre à cette version. Les sorties du 05 ne sont pas réutilisées.

**Simulation EPI révisée** : `SIMULATION_MODEL="auto"` sélectionne un modèle de
voxels finis pour les acquisitions réelles, avec un B0 local pondéré par les échos.
Le signal est intégré sur la surface de chaque pixel en tenant compte des gradients
B0 et ACDC ; ce même modèle et son gradient sont utilisés pendant l'optimisation.
Pour les jeux synthétiques issus du générateur ponctuel, le mode automatique conserve
le modèle ponctuel. `"point"` et `"finite_voxel"` permettent des comparaisons explicites.
Après une mise à jour des modules, redémarrer le noyau et relancer depuis les imports.


**Modèle temporel complexe révisé** : le mode automatique ajuste les 32 échos complexes
et les 32 récepteurs, avec covariance du bruit. Une décroissance exponentielle,
une dispersion gaussienne ou une dispersion uniforme (sinc signé) sont comparées
sur les derniers échos réservés, puis le modèle choisi est ajusté aux 32 échos.
Les zéros et échos faibles restent des observations : ils ne remettent plus la décroissance à zéro.
L’enveloppe et la décroissance résiduelle sont séparées des gradients dans le plan,
qui restent intégrés une seule fois dans MRzero.

La dispersion résiduelle est effective et sans direction spatiale identifiée.
Elle reste dans le diagnostic simulé B0 nul : on ne suppose pas que les bobines pourront la
réphaser. La prédiction des trous MEGRE et celle de l’EPI sont évaluées séparément.
La reconstruction complexe, l’ajustement et le modèle utilisent CUDA quand demandé.
**Référence au choix** : `REFERENCE_SCAN_PATH` accepte n'importe quel nom de fichier
Twix, y compris les nouvelles références 64×64 et 128×128. `REFERENCE_SEQUENCE_NAME`
permet de fournir sa propre séquence ; sans séquence compatible, le lecteur vérifie
une acquisition cartésienne complète dans le Twix et signale sa trajectoire nominale.
Le FOV, la matrice native, le TE et le TR restent ceux de chaque acquisition.
La référence et les 32 échos contribuent au modèle d'entrée MRzero : structure
GRE, contraste inter-acquisitions lisse et dynamique complexe multi-échos.
Indépendamment de cette construction, la GRE mesurée est la cible de l'étape 6 :
l'EPI simulée avec les courants ACDC doit lui ressembler. La simulation EPI B0 nul
reste disponible comme diagnostic physique.


In [ ]:
from pathlib import Path
from copy import deepcopy
from datetime import datetime
import hashlib, json, sys, time
import numpy as np
import matplotlib.pyplot as plt
import torch
from IPython.display import display, Markdown

PROJECT_ROOT=Path.cwd() if (Path.cwd()/"Experiment").is_dir() else Path.cwd().parent
for folder in (PROJECT_ROOT/"Experiment",PROJECT_ROOT/"MRzero_validation"):
    if str(folder) not in sys.path:sys.path.insert(0,str(folder))
from twin_inputs import read_twin_inputs,prepare_echo_object
from echo_shim_inputs import prepare_echo_shim_inputs
from experiment_echo import EchoSubspinACDCModel
from experiment_voxel import FiniteVoxelEchoACDCModel
from experiment_multiecho import MultiEchoVoxelACDCModel
from multiecho_object import prepare_complex_echo_model,multiecho_comparison
from reference_echo_prior import apply_reference_amplitude_prior
from measured_image_target import configure_measured_gre_target,relative_image_mse
from b0_regularization import regularize_b0_local_plane
from inplane_geometry import inplane_matrix,FrameReconstruction
from epi_twin import NativeEPITwin
from twin_geometry import compare_magnitude
from acdc_gpu_field import solve_field_gpu
from acdc_gpu import response_matrices,select_device
from experiment_optimization import optimize_mrzero_image
from field_guidance import FieldGuidance
from live_optimization import LiveOptimizationPlot
from optimization_image_history import OptimizationImageHistory
from acdc_waveform_plot import plot_acdc_waveform
from measured_scan_comparison import compare_measured_scan_set
from export_acdc_waveform import quantize_currents,export_waveform,export_zero_time_copy,validate_waveform_settings
from shim_export_timing import sequence_trigger_timing,waveform_edges_for_origin
from field_shim_export import export_field_shim_waveforms
from prepare_acdc_inputs import read_scan
from read_acdc_b0 import read_b0_sequence
from read_acdc_sequence import read_sequence

INPUT_DIR=PROJECT_ROOT/"acquisition_021_(06)"
# Séquence EPI à corriger : le scan EPI mesuré doit lui correspondre.
GRE_EPI_SEQUENCE_NAME="gre_epi_256x256_fov200mm_nav3_dwell5us_rise240us_ext1_single_trigger_first_readout_v5.seq"

GRE_EPI_SCAN_PATH="meas_MID00756_FID48499_scan256.dat"  # Un nom .dat libre, relatif à INPUT_DIR, ou un chemin absolu.
# None recherche un seul *scan.dat, hors fichiers référence et B0 sélectionnés.

# Référence GRE indépendante de la matrice EPI (32², 64², 128²...).
REFERENCE_SCAN_PATH="meas_MID00758_FID48501_reference128.dat"  # Nom .dat libre, ou None pour auto.
# Exemple 64² : "meas_MID00757_FID48500_reference64.dat"
# Exemple 128² : "meas_MID00758_FID48501_reference128.dat"
REFERENCE_SEQUENCE_NAME=None  # .seq correspondant, ou None pour rechercher une GRE compatible.
# Sans .seq compatible : reconstruction cartésienne nominale vérifiée dans le Twix.
# Un .seq explicite incompatible provoque une erreur ; aucune conversion implicite.

# Acquisition multi-échos indépendante pour B0 / amplitude / relaxation.
B0_SEQUENCE_NAME="megre_b0_s0_t2star_128x128_fov200mm_32echo.seq"
B0_SCAN_PATH=None  # .dat 32 échos relatif à INPUT_DIR, ou chemin absolu.
# None recherche un seul *b0_map.dat, dont les 32 échos sont ensuite vérifiés.
# Les trois noms *_SEQUENCE_NAME acceptent aussi un chemin .seq absolu.



INPUT_DIR=PROJECT_ROOT/"acquisition_027_(06)_fov250mm"
# Séquence EPI à corriger : le scan EPI mesuré doit lui correspondre.
GRE_EPI_SEQUENCE_NAME="gre_epi_128x128_fov250mm_nav3_cimax_snr_t2star40ms_line_triggers_v6.seq"

GRE_EPI_SCAN_PATH=None  # Un nom .dat libre, relatif à INPUT_DIR, ou un chemin absolu.
# None recherche un seul *scan.dat, hors fichiers référence et B0 sélectionnés.

# Référence GRE indépendante de la matrice EPI (32², 64², 128²...).
REFERENCE_SCAN_PATH=None # Nom .dat libre, ou None pour auto.
# Exemple 64² : "meas_MID00757_FID48500_reference64.dat"
# Exemple 128² : "meas_MID00758_FID48501_reference128.dat"
REFERENCE_SEQUENCE_NAME="gre_128x128_fov250mm_flip15deg_tr15ms.seq"  # .seq correspondant, ou None pour rechercher une GRE compatible.
# Sans .seq compatible : reconstruction cartésienne nominale vérifiée dans le Twix.
# Un .seq explicite incompatible provoque une erreur ; aucune conversion implicite.

# Acquisition multi-échos indépendante pour B0 / amplitude / relaxation.
B0_SEQUENCE_NAME="megre_b0_s0_t2star_128x128_fov250mm_32echo.seq"
B0_SCAN_PATH=None  # .dat 32 échos relatif à INPUT_DIR, ou chemin absolu.
# None recherche un seul *b0_map.dat, dont les 32 échos sont ensuite vérifiés.
# Les trois noms *_SEQUENCE_NAME acceptent aussi un chemin .seq absolu.

OUTPUT_DIR=PROJECT_ROOT/"MRzero_validation/results_experiment_echo32"/datetime.now().strftime("mrzero06_%Y%m%d_%H%M%S_%f")

# Calibration commune : même fichier que /homes/7/mv1289/MAURICE/.../data/ACDC_3T.npz.
CALIBRATION_PATH=PROJECT_ROOT/"data/ACDC_3T.npz"
WAVEFORM_OUTPUT_DIR=INPUT_DIR/"waveform"
OUTPUT_DIR=PROJECT_ROOT/"MRzero_validation/results_experiment_echo32"/datetime.now().strftime("mrzero06_%Y%m%d_%H%M%S_%f")
print("Calibration commune :",CALIBRATION_PATH.resolve())
print("Dossier des waveforms :",WAVEFORM_OUTPUT_DIR.resolve())





## 1. Paramètres à modifier

**Canaux hors service** : `DISABLED_CHANNELS=[]` conserve toutes les voies.
Par exemple, `[7]` désactive la sortie physique 7, et `[7, 12]` en désactive deux.
Les numéros sont ceux de `WAVEFORM_PARAMETERS["channel_map"]` (1 à 32 par défaut),
pas les indices Python ; la sortie 7 correspond à `ch6_A` dans le CSV.
Leur courant est contraint à zéro dès le shim initial et pendant les optimisations
image, statique et dynamique. Les autres voies sont réoptimisées en conséquence.
Les colonnes sont conservées et l’export refuse un courant non nul sur une voie désactivée.
Après modification, relancer depuis cette cellule pour reconstruire le modèle.

**Orientation données → bobines** : `ALIGNMENT_ANGLE_DEG` donne l'angle autour de Z,
en degrés positifs de X vers Y. `ALIGNMENT_MIRROR_X` remplace X par −X ;
`ALIGNMENT_MIRROR_Y` remplace Y par −Y. Les miroirs sont appliqués **avant** la rotation.
L'amplitude, le B0 et les ADC suivent ensemble cette orientation. Les bobines restent
dans leur repère de calibration : changer cet alignement adapte donc les courants.
La valeur initiale 180° reprend ton orientation préférée.

**Repère commun facultatif** : les paramètres `FRAME_*` ne changent que la représentation
commune après cet alignement. Ils ne changent pas le placement relatif aux bobines.
Changer seulement ce repère doit conserver les courants physiques.

Exemples : 90° = quart de tour ; −30° = rotation horaire ; un miroir X seul =
`ANGLE=0, MIRROR_X=True, MIRROR_Y=False`. Deux miroirs équivalent à une rotation de 180°.
Après une modification, relancer le notebook depuis cette cellule pour recalculer
les positions, les champs et les courants correspondants.
Le dossier d'acquisition et la séquence sont choisis en tête du notebook avec
`INPUT_DIR` et `GRE_EPI_SEQUENCE_NAME`. Le choix actuel est l'acquisition 021_(06),
GRE-EPI 128×128. Le nom est explicite même si plusieurs `.seq` sont présents.
Le scan brut doit correspondre à cette séquence ; changer le nom ne convertit
pas une acquisition 64×64 en 128×128. Après un changement, relancer depuis les imports.


**Cartographie multi-échos** : `B0_SEQUENCE_NAME` sélectionne le `.seq` de cartographie,
`B0_SCAN_PATH` son acquisition Twix. Le nombre d'échos, TE, TR, dwell, FOV et matrice
sont vérifiés. `B0_FIELD_ECHO_COUNT=6` réserve les six premiers échos (jusqu'à 11,223 ms)
au calcul de B0 ; les 32 restent disponibles pour ajuster R2*.

**Bruit et support** : inspecter les coins définis par `BACKGROUND_CORNER_M` et les
masques affichés. `NOISE_MASK=None` utilise ces coins ; un masque booléen natif peut
être fourni explicitement. Ils doivent être sans signal objet. Le seuil SNR est
appliqué après retrait de la puissance de bruit RSS. La calibration est requise par
défaut ; un fond contaminé ou absent donne une erreur explicite, plutôt qu'un SNR inventé.

**Calcul CPU/GPU** : `GPU_PROFILE="auto"` choisit la carte disponible et
`GPU_REQUIRED=True` impose une exécution CUDA réelle. Le contrôle en début de
notebook affiche l'interpréteur, PyTorch, CUDA et le périphérique ; un échec
arrête le calcul avant la préparation des données. Le modèle et l'étape 6
vérifient aussi cette exigence, y compris si un ancien modèle CPU reste en mémoire.
Pour un calcul CPU volontaire, choisir `GPU_REQUIRED=False` et `GPU_PROFILE="cpu"`.
Les profils ne changent pas la distribution PyTorch installée : installer une
version compatible dans le Python du notebook puis redémarrer le noyau ; voir
`../docs/SERVER_SETUP.md`. Changer seulement ces paramètres ne déplace pas un
modèle déjà construit : relancer depuis le début.

Pour une distribution commune à la RTX PRO 6000 Blackwell, à la GTX 1660 Ti
et à la V100, installer une fois dans le noyau voulu :
`%pip install --upgrade torch==2.8.0+cu128 --index-url https://download.pytorch.org/whl/cu128`,
puis redémarrer le noyau. Cette distribution Linux couvre `sm_120`, `sm_75`
et `sm_70` ; elle demande aussi un pilote NVIDIA compatible CUDA 12.8.
Une V100 peut conserver son environnement CUDA 11.8 sur sa machine si nécessaire.
La variante CUDA 12.8 de PyTorch 2.7.1 excluait `sm_70` ; la version 2.8 le rétablit.

**Comparaison historique** : avec `ECHO_SIGNAL_MODEL="legacy"`, `RATE_MODE="guarded"` applique les ajustements locaux fiables à TE EPI.
`"pooled"` sélectionne le taux global conservateur ; `"none"` désactive la décroissance.
Les critères actuels demandent un SNR suffisant à tous les échos. Des voxels aux derniers
échos trop faibles peuvent donc être refusés ; leurs replis sont affichés. Le taux et
l'amplitude du fit sont effectifs : ils ne constituent pas une carte de densité protonique.

**Signal complexe des 32 échos** : `ECHO_SIGNAL_MODEL="auto"` active le nouveau fit
pour le mesuré ; `"complex"` le demande explicitement et `"legacy"` conserve la
comparaison historique. `RATE_MODE` sert au mode historique ; le nouveau mode emploie
`effective_rate_mode="complex"`. Le B0 précoce initialise le fit ; les 32 échos
peuvent ensuite affiner sa fréquence centrale. Les pentes dans le plan restent
celles fixées avant le fit pour éviter un double comptage du déphasage.

Le fit compare trois familles sur les `COMPLEX_VALIDATION_ECHOES` derniers échos,
sans réajuster les amplitudes à ces échos de contrôle. Le modèle retenu utilise
ensuite tous les échos. La qualité insuffisante et les limites de paramètres restent
signalées, sans supprimer les trous ni rétablir une décroissance nulle.
Ces contrôles restent internes à l’acquisition : les temps EPI au-delà du dernier
écho impliquent toujours une extrapolation.
**Choix indépendants des acquisitions** : `GRE_EPI_SCAN_PATH`, `REFERENCE_SCAN_PATH`
et `B0_SCAN_PATH` acceptent chacun un nom relatif à `INPUT_DIR` ou un chemin absolu.
Renseigner `REFERENCE_SCAN_PATH` avec le nouveau `.dat` 64×64 ou 128×128 suffit ;
aucun suffixe particulier n'est requis. Avec `None`, la référence historique est
recherchée via un unique `*reference*.dat`. Une sélection ambiguë provoque une erreur.
Les trois acquisitions gardent leurs grilles physiques natives et doivent couvrir
la même coupe ; une référence 64×64 n'est jamais assimilée à une mesure 128×128.
Les `*_SEQUENCE_NAME` désignent les fichiers du sous-dossier `sequences` ou des
chemins absolus. Le `.seq` de référence est facultatif ; un `.seq` explicite est
contrôlé strictement contre le `.dat` choisi.

**Objet MRzero construit avec la GRE et les 32 échos** :
`OBJECT_AMPLITUDE_SOURCE="joint"` utilise la structure de la GRE comme candidat
pour l'enveloppe du signal, avec une correction positive et spatialement lisse
du contraste GRE→MEGRE. `OBJECT_CONTRAST_SIGMA_MM` fixe son échelle physique.
Cette correction et les paramètres temporels sont estimés sans les derniers
échos réservés ; ceux-ci choisissent le poids du candidat. Sans amélioration
mesurée, le modèle conserve son amplitude multi-échos et le rapport le signale.
La GRE n'est jamais réécrite par cette calibration : elle reste la cible mesurée
indépendante de l'étape 6. B0, décroissance et cohérence viennent des 32 échos.

`"echo32"` conserve explicitement l'enveloppe multi-échos, `"auto"` teste le
candidat historique à gain global seul et `"reference"` impose ce candidat.
Une GRE unique ne mesure ni PD ni T1. La correction lisse est un contraste
inter-acquisitions effectif, sans identification séparée de T1, B1 ou réception.
Interpoler une GRE64 sur la grille EPI128 ne crée pas de détails acquis à 128².

**Cible de l'optimisation image** : la GRE mesurée sélectionnée est utilisée
quelle que soit sa matrice, après interpolation dans les coordonnées physiques
communes. `OPTIMIZATION_REFERENCE_THRESHOLD=0` compare toute la couverture
commune, y compris les pixels sombres. Une valeur positive définit une ROI GRE
fixe. Les gains simulation→EPI sans ACDC et EPI sans ACDC→GRE sont estimés une
seule fois, puis gelés. Aucun gain ni recalage n'est réajusté par itération.
La simulation B0 nul reste un diagnostic ; elle ne participe pas à cette cible.

La calibration est partagée dans `PROJECT_ROOT/data/ACDC_3T.npz` via
`CALIBRATION_PATH`. Les CSV sont enregistrés dans `INPUT_DIR/waveform`.
`EXPORT_FIELD_SHIM_WAVEFORMS` active les exports statique et dynamique
indépendamment de l’optimisation image.


In [ ]:
# Alignement physique des données d'imagerie dans le repère des bobines ACDC.
ALIGNMENT_ANGLE_DEG=90
ALIGNMENT_MIRROR_X=False
ALIGNMENT_MIRROR_Y=False

# Changement de représentation commun, sans déplacement relatif aux bobines.
FRAME_ANGLE_DEG=0.0
FRAME_MIRROR_X=False
FRAME_MIRROR_Y=False

# Objet et relaxation issus des 32 échos.
OBJECT_THRESHOLD_FRACTION=0.04
BACKGROUND_CORNER_M=0.085
NOISE_MASK=None                 # Masque booléen 128² facultatif, explicitement sans signal.
OBJECT_MIN_SNR=5.0
OBJECT_MIN_COMPONENT_VOXELS=4
REQUIRE_NOISE_CALIBRATION=True  # False autorise le repli sans relaxation si bruit indisponible.
B0_FIELD_ECHO_COUNT=6           # Initialisation B0 ; le fit complexe utilise ensuite les 32 échos.
ECHO_SIGNAL_MODEL="auto"        # "auto" : complexe pour le mesuré ; "legacy" : ancien fit RSS.
COMPLEX_FIT_BATCH_SIZE=256
COMPLEX_FIT_ITERATIONS=60
COMPLEX_VALIDATION_ECHOES=6      # Derniers échos réservés pour choisir le modèle, puis refit des 32.
COMPLEX_MAX_WIDTH_HZ=120.        # Borne fixée avant le fit, non ajustée sur l'EPI.
OBJECT_AMPLITUDE_SOURCE="joint"  # GRE + 32 échos, contraste lisse et choix sur échos réservés.
OBJECT_CONTRAST_SIGMA_MM=10.0    # Échelle du contraste inter-acquisitions, fixée avant l'ajustement.
REFERENCE_PRIOR_MIN_VALIDATION_IMPROVEMENT=0.02  # Gain minimal sur les échos réservés.
SIMULATION_MODEL="auto"         # "auto", "point" ou "finite_voxel".
B0_WEIGHTING="auto"             # "auto", "uniform" ou "echo_power".
BASELINE_CALIBRATION="auto"   # "auto" (mesuré seulement), "off" ou "on".
CALIBRATION_BOUNDS={"frequency_offset_Hz":30.,"readout_shift_mm":2.,"odd_even_phase_rad":.15}
COMPARE_POINT_BASELINE=True     # Comparer l'ancien modèle au scan réel avant optimisation.
B0_MODE="regularized"          # "regularized" ou "raw".
B0_SIGMA_PIXELS=1.0             # Échelle native fixée, sans ajustement au scan EPI.
RATE_MODE="guarded"             # "pooled", "guarded" ou "none".
SUBSPINS_Z=8                   # 128² dans le plan, puis sous-spins dans les 5 mm.

GPU_PROFILE="auto"             # Détecte GTX 1660 Ti, V100 et Blackwell sans changer le notebook.
GPU_REQUIRED=True              # Arrête le calcul si CUDA ne fonctionne pas. False autorise le CPU.
GPU_PRECISION="auto"
GPU_MEMORY_FRACTION=0.60         # Favorise le cache MRzero sur GPU ; blocs adaptés à la mémoire libre.
SAMPLE_CHUNK=None
RECONSTRUCTION_CHUNK=256

MAX_CURRENT_A=3.0
DISABLED_CHANNELS=[9]  # Sorties physiques hors service (1..32) ; ex. [7] ou [7, 12].
TOTAL_CURRENT_LIMIT_A=30.0
FIELD_LIMIT_HZ=1000.0
CURRENT_RESPONSE={"enabled":True,"tau_s":133e-6/np.log(100.)}
HARDWARE={"min_update_interval_s":8e-6,"dac_resolution_A":0.0006,
          "max_slew_A_per_s":None,"avg_abs_current_A":None,
          "coil_R_ohm":None,"coil_L_H":None,"max_voltage_V":None,"max_coil_power_W":None}

IMAGE_INITIALIZATION="static"  # "static" ou "zero".
# Guidage physique pour les fortes distorsions/repliements EPI.
FIELD_GUIDANCE_WEIGHT=0.05      # Point de départ à comparer ; 0 retrouve le coût image seul.
FIELD_GUIDANCE_SCALE_HZ=100.0   # Normalisation fixe du résidu B0, pas une limite de champ.
MAX_ITERATIONS=1000
LEARNING_RATE_A=0.001
PATIENCE=100
TARGET_MSE=None
OPTIMIZATION_REFERENCE_THRESHOLD=0.0  # 0 : couverture GRE/EPI commune, pixels sombres inclus.
# >0 : ROI au-dessus de cette fraction du maximum GRE ; aucun masque dépendant des courants.
LIVE_UPDATE_EVERY=20           # Réduit le coût CPU des figures/HTML pendant le calcul GPU.
RUN_OPTIMIZATION=True

# Export local seulement ; le contrôleur est utilisé séparément.
EXPORT_WAVEFORM=True         # Waveform de l’optimisation image.
EXPORT_FIELD_SHIM_WAVEFORMS=True  # Statique et dynamique, même sans optimisation image.
FIELD_SHIM_RASTER_S=10e-6
FIELD_SHIM_MAX_ITERATIONS=80
CSV_TIME_ORIGIN="first_imaging_trigger"  # Ou "rf".
TRIGGER_CHANNEL="ext1"

# Mesure des courants : une colonne de temps, remise à zéro à chaque trigger.
EXPORT_ADC_WAVEFORM=True
WAVEFORM_ADC_INTERVAL_S=0.000050
WAVEFORM_ADC_DURATION_S=None  # Durée totale depuis le premier trigger ; None = fin du dernier readout.
WAVEFORM_PARAMETERS={"channel_map":list(range(1,33)),"channel_polarity":[-1]*32,
    "max_current_A":MAX_CURRENT_A,"total_current_limit_A":TOTAL_CURRENT_LIMIT_A,
    "min_dac_interval_s":8e-6,"disabled_channels":DISABLED_CHANNELS}
# Une seule configuration pour les contraintes du modèle et tous les exports.
WAVEFORM_PARAMETERS=validate_waveform_settings(WAVEFORM_PARAMETERS,len(WAVEFORM_PARAMETERS["channel_map"]))
CHANNEL_MAX_CURRENT_A=np.asarray(WAVEFORM_PARAMETERS["max_current_A"],dtype=float)

CALIBRATION_PARAMETERS={"units":"Hz/A","registration_confirmed":True,
    "patient_to_source_m":np.eye(4).tolist(),
    "source_sha256":"bace6a584e780d80af8f2ca7c12957bd5b9dc7462be0968f949cc9a99243a06c",
    "registration_note":"Placement Bay5 confirmé précédemment ; orientation d'imagerie réglée ci-dessus."}
# Comparaison finale : None prend tous les *scan.dat du dossier results.
# Pour choisir : ["nom_du_scan_corrige.dat"].
CORRECTED_SCAN_NAMES=None
NORMALIZE_COMPARISON=True
COMPARISON_REFERENCE_THRESHOLD=0.1  # ROI GRE ; 0 pour toute la couverture commune.
WAVEFORM_PREVIEW_CHANNEL=None      # Sortie physique (1..32), ou plus grand saut.

In [ ]:
def table(rows,headings=("Paramètre","Valeur")):
    display(Markdown("| "+" | ".join(headings)+" |\n| "+" | ".join(["---"]*len(headings))+" |\n"+
        "\n".join("| "+" | ".join(str(v) for v in row)+" |" for row in rows)))

def show(fig):
    plt.show();plt.close(fig)

def image_physical(ax,image,X,Y,title,cmap="gray",**options):
    artist=ax.pcolormesh(X*1000,Y*1000,np.asarray(image),shading="auto",cmap=cmap,**options)
    ax.set(title=title,xlabel="X [mm]",ylabel="Y [mm]",aspect="equal")
    return artist

def json_safe(value):
    if isinstance(value,dict):return {str(k):json_safe(v) for k,v in value.items()}
    if isinstance(value,(list,tuple)):return [json_safe(v) for v in value]
    if isinstance(value,np.ndarray):return json_safe(value.tolist())
    if isinstance(value,np.generic):return json_safe(value.item())
    if isinstance(value,Path):return str(value)
    if isinstance(value,float) and not np.isfinite(value):return None
    return value

angles={"alignment_angle_deg":ALIGNMENT_ANGLE_DEG,"alignment_mirror_x":ALIGNMENT_MIRROR_X,
    "alignment_mirror_y":ALIGNMENT_MIRROR_Y,"frame_angle_deg":FRAME_ANGLE_DEG,
    "frame_mirror_x":FRAME_MIRROR_X,"frame_mirror_y":FRAME_MIRROR_Y}
Q=inplane_matrix(ALIGNMENT_ANGLE_DEG,ALIGNMENT_MIRROR_X,ALIGNMENT_MIRROR_Y)
P=inplane_matrix(FRAME_ANGLE_DEG,FRAME_MIRROR_X,FRAME_MIRROR_Y)
O=P@Q
table(list(angles.items()))
table([("Sorties physiques hors service",WAVEFORM_PARAMETERS["disabled_channels"]),
       ("Nombre de voies actives",int(np.count_nonzero(CHANNEL_MAX_CURRENT_A)))])
print("Matrice données→bobines Q :\n",Q,"\nMatrice du repère de calcul O=P@Q :\n",O)
print("Sorties :",OUTPUT_DIR)

# Vérifier le GPU avant la lecture et la préparation des acquisitions.
print("Python du noyau :",sys.executable)
print("PyTorch / runtime CUDA :",torch.__version__,"/",torch.version.cuda)
execution_device,execution_dtype,execution_report=select_device(
    GPU_PROFILE,GPU_PRECISION,require_cuda=GPU_REQUIRED)
table([("Périphérique de calcul",str(execution_device)),
       ("GPU",execution_report["gpu_name"]),
       ("Précision",execution_report["precision"]),
       ("GPU obligatoire",GPU_REQUIRED)])


## 2. Vérifier la séquence et préparer les 32 échos

Le contrôle ci-dessous lit les événements Pulseq réels avant les données Twix.
Il affiche le nombre de points ADC, les TE, le TR et les temps EPI. Dans le fichier
fourni, la définition accessoire `EchoSamples` annonce 64, alors que les événements
ADC et `Matrix` indiquent bien 128 : le lecteur vérifie les événements réels.
Les petits décalages de kx entre échos sont conservés dans la reconstruction complexe.

La lecture suivante vérifie la correspondance du `.dat` avec cette séquence.
Le bruit est mesuré **avant** le masque objet, dans les seuls coins ou le masque
explicitement choisi. Aucune région n'est remplacée automatiquement par le complément
de l'objet. Les figures permettent de vérifier le bruit, le support et la couverture B0.
Les phases B0 utilisent les échos précoces ; l'ajustement de relaxation conserve les 32.

La référence GRE sélectionnée est affichée dès la lecture, dans ses coordonnées
physiques natives, avec son fichier, sa matrice, son TE et son TR, avant les
ajustements des 32 échos.


In [ ]:
import warnings
import importlib
import read_acdc_b0 as b0_reader
# Le noyau peut avoir chargé l'ancien lecteur depuis le 05.
b0_reader=importlib.reload(b0_reader)
read_b0_sequence=b0_reader.read_b0_sequence

def selected_sequence_path(name):
    if name is None:
        return None
    path=Path(name)
    return path if path.is_absolute() else INPUT_DIR/"sequences"/path

selected_epi_sequence=selected_sequence_path(GRE_EPI_SEQUENCE_NAME)
selected_b0_sequence=selected_sequence_path(B0_SEQUENCE_NAME)
selected_reference_sequence=selected_sequence_path(REFERENCE_SEQUENCE_NAME)

with warnings.catch_warnings():
    warnings.filterwarnings("ignore", message=r"^File version 1\.5\.1 is higher than installed package version 1\.5\.0\. This may cause parsing errors\.$",
                            category=UserWarning, module=r"^pypulseq\.Sequence\.read_seq$")
    b0_sequence_report=read_b0_sequence(selected_b0_sequence)
    epi_preview,_=read_sequence(selected_epi_sequence)
if b0_sequence_report["echo_count"] != 32 or tuple(b0_sequence_report["image_shape"]) != (128,128):
    raise ValueError("Le notebook 06 attend une cartographie de 32 échos, matrice 128×128.")
if not 2 <= B0_FIELD_ECHO_COUNT <= b0_sequence_report["echo_count"]:
    raise ValueError("B0_FIELD_ECHO_COUNT doit sélectionner entre 2 et 32 échos.")
expected_echo_times=np.asarray(b0_sequence_report["TE_s"])
epi_relative_times=np.asarray(epi_preview["time_s"])-np.asarray(epi_preview["rf_time_s"])
sequence_assessment={"B0_sequence":B0_SEQUENCE_NAME,"B0_expected_echo_count":32,
    "B0_field_echo_count":B0_FIELD_ECHO_COUNT,"echo_times_s":expected_echo_times,
    "target_TE_s":float(epi_preview["TE_s"]),
    "EPI_ADC_range_s":[float(epi_relative_times.min()),float(epi_relative_times.max())],
    "extrapolation_to_TE_s":max(0.,float(epi_preview["TE_s"]-expected_echo_times[-1])),
    "extrapolation_to_last_ADC_s":max(0.,float(epi_relative_times.max()-expected_echo_times[-1]))}
table([("Séquence cartographie",B0_SEQUENCE_NAME),
       ("Échos / matrice ADC",f"{b0_sequence_report['echo_count']} / {b0_sequence_report['image_shape']}"),
       ("Premier / dernier TE [ms]",expected_echo_times[[0,-1]]*1000),
       ("TR cartographie [ms]",b0_sequence_report["TR_s"]*1000),
       ("Durée cartographie [s]",b0_sequence_report["sequence_duration_s"]),
       ("TE EPI [ms]",epi_preview["TE_s"]*1000),
       ("Premier / dernier ADC après RF [ms]",np.array(sequence_assessment["EPI_ADC_range_s"])*1000),
       ("Extrapolation dernier écho → TE EPI [ms]",sequence_assessment["extrapolation_to_TE_s"]*1000),
       ("Extrapolation dernier écho → dernier ADC [ms]",sequence_assessment["extrapolation_to_last_ADC_s"]*1000)])
print("Twix cartographie :",B0_SCAN_PATH if B0_SCAN_PATH is not None else "détection d'un unique *b0_map.dat ; 32 échos requis")
print("Twix référence :",REFERENCE_SCAN_PATH if REFERENCE_SCAN_PATH is not None else "unique *reference*.dat")
print("Séquence référence :",selected_reference_sequence or "auto compatible, sinon cartésien Twix nominal")
print("Twix EPI :",GRE_EPI_SCAN_PATH if GRE_EPI_SCAN_PATH is not None else "unique *scan.dat hors référence/B0")


In [ ]:
# Actualiser le lecteur après un changement de code dans le même noyau.
import importlib
import twin_inputs
twin_inputs=importlib.reload(twin_inputs)
read_twin_inputs=twin_inputs.read_twin_inputs
prepare_echo_object=twin_inputs.prepare_echo_object

with warnings.catch_warnings():
    warnings.filterwarnings("ignore", message=r"^File version 1\.5\.1 is higher than installed package version 1\.5\.0\. This may cause parsing errors\.$",
                            category=UserWarning, module=r"^pypulseq\.Sequence\.read_seq$")
    inputs=read_twin_inputs(INPUT_DIR,
        scan_sequence_path=selected_epi_sequence,scan_path=GRE_EPI_SCAN_PATH,
        reference_sequence_path=selected_reference_sequence,reference_scan_path=REFERENCE_SCAN_PATH,
        b0_sequence_path=selected_b0_sequence,
        b0_scan_path=B0_SCAN_PATH,b0_echo_count=B0_FIELD_ECHO_COUNT,
        reconstruction_device=execution_device,reconstruction_dtype=execution_dtype)
# Afficher la référence sélectionnée avant l'ajustement des 32 échos.
print("Référence GRE :",inputs["paths"]["reference"].resolve())
print("Séquence GRE réellement utilisée :",inputs["paths"].get("reference_sequence","aucune — trajectoire cartésienne nominale Twix"))
table([("Matrice référence native",np.asarray(inputs["gre"]["magnitude"]).shape),
       ("TE référence [ms]",float(inputs["gre"].get("TE_s",np.nan))*1000),
       ("TR référence [ms]",float(inputs["gre"].get("TR_s",np.nan))*1000),
       ("Validation référence",inputs.get("reference_sequence_validation",{})),
       ("Contrôles de coupe",inputs.get("geometry_checks",[]))])
reference_gre=inputs["gre"]
reference_magnitude=np.asarray(reference_gre["magnitude"])
reference_rows,reference_columns=reference_magnitude.shape
fig,ax=plt.subplots(figsize=(8,6),layout="constrained")
reference_artist=image_physical(ax,reference_magnitude,
    reference_gre["X_m"],reference_gre["Y_m"],
    f"Référence GRE mesurée — {reference_rows}×{reference_columns}\n"
    f"{inputs['paths']['reference'].name}",vmin=0)
fig.colorbar(reference_artist,ax=ax,label="Magnitude RSS [u.a.]")
show(fig)

if len(inputs["echo_times_s"]) != 32:
    raise ValueError("Le fichier Twix sélectionné ne contient pas les 32 échos attendus.")
fit=prepare_echo_object(inputs,OBJECT_THRESHOLD_FRACTION,BACKGROUND_CORNER_M,
    allow_uncalibrated_noise=not REQUIRE_NOISE_CALIBRATION,noise_mask=NOISE_MASK,
    support_min_snr=OBJECT_MIN_SNR,min_component_voxels=OBJECT_MIN_COMPONENT_VOXELS)
# Le support et le bruit restent indépendants du nouveau modèle temporel.
legacy_echo_fit=fit
if ECHO_SIGNAL_MODEL not in {"auto","complex","legacy"}:
    raise ValueError("ECHO_SIGNAL_MODEL doit valoir auto, complex ou legacy.")
input_is_synthetic=(inputs.get("manifest") or {}).get("format") in {
    "mrzero_synthetic_acquisition","mrzero_synthetic_twix"}
complex_echo_enabled=(ECHO_SIGNAL_MODEL=="complex" or
                      (ECHO_SIGNAL_MODEL=="auto" and not input_is_synthetic))
effective_rate_mode="complex" if complex_echo_enabled else RATE_MODE
complex_echo_diagnostics=None
if complex_echo_enabled:
    echo_weighting=("echo_power" if B0_WEIGHTING=="auto" else B0_WEIGHTING)
    fit=prepare_complex_echo_model(inputs,legacy_echo_fit,
        device=str(execution_device),require_cuda=GPU_REQUIRED,
        b0_sigma_pixels=B0_SIGMA_PIXELS,b0_weighting=echo_weighting,
        batch_size=COMPLEX_FIT_BATCH_SIZE,max_iterations=COMPLEX_FIT_ITERATIONS,
        max_width_Hz=COMPLEX_MAX_WIDTH_HZ,validation_echo_count=COMPLEX_VALIDATION_ECHOES)
    fit=apply_reference_amplitude_prior(inputs,fit,
        source=OBJECT_AMPLITUDE_SOURCE,contrast_sigma_mm=OBJECT_CONTRAST_SIGMA_MM,
        device=str(execution_device),require_cuda=GPU_REQUIRED,
        validation_echo_count=COMPLEX_VALIDATION_ECHOES,
        min_relative_validation_improvement=REFERENCE_PRIOR_MIN_VALIDATION_IMPROVEMENT,
        fit_options={"batch_size":COMPLEX_FIT_BATCH_SIZE,"max_iterations":COMPLEX_FIT_ITERATIONS,
                     "max_width_Hz":COMPLEX_MAX_WIDTH_HZ})
    table(list(fit["report"]["reference_amplitude_prior"].items()),("Apport de la référence à l'amplitude","Valeur"))
    complex_echo_diagnostics=multiecho_comparison(inputs,fit)
    table(list(fit["report"]["multi_echo"].items()),("Fit complexe des 32 échos","Valeur"))
    table([(row["region"],row["pixels"],row["legacy_RSS_RMSE_over_first_echo"],
            row["complex_RSS_RMSE_over_first_echo"]) for row in complex_echo_diagnostics["rows"]],
          ("Région MEGRE","Pixels","Erreur ancien modèle / écho 1","Erreur complexe / écho 1"))
print("Modèle temporel utilisé par MRzero :",effective_rate_mode)
noise_info=fit["report"]["noise"]
support_info=fit["report"]["support"]
raw=inputs["b0_raw"]
acquisition_native=inputs["acquisition"]
pooled=fit["pooled"]
print("Cartographie Twix :",inputs["paths"]["b0"].resolve())
print("Séquence cartographie :",inputs["paths"]["b0_sequence"].resolve())
print("Scan EPI sans ACDC :",inputs["paths"]["scan"].resolve())
print(f"Bruit : {noise_info['available_sample_count']} vecteurs complexes finis / "
      f"{noise_info['minimum_sample_count']} requis ; calibration = {noise_info['calibrated']}.")
if not noise_info["calibrated"]:
    print("Repli : premier écho, sans décroissance R2* supplémentaire. Motif :",noise_info["fallback_reason"])
table([("Nombre d'échos lus pour relaxation",len(inputs["echo_times_s"])),
       ("Nombre d'échos précoces pour B0",inputs["field_echo_count"]),
       ("TE cartographie [ms]",inputs["echo_times_s"]*1000),
       ("TE EPI [ms]",acquisition_native["TE_s"]*1000),
       ("Pixels bruit indépendants",int(fit["noise_mask"].sum())),
       ("Pixels support objet",int(fit["object_support"].sum())),
       ("Pixels simulables avec B0",int(fit["simulation_support"].sum())),
       ("Taux global R2* diagnostique [s⁻¹]",pooled.get("R2star_sInv")),
       ("Incertitude du taux global [s⁻¹]",pooled.get("R2star_std_sInv")),
       ("Taux global accepté à TE EPI",pooled.get("target_reliable_mask",False)),
       ("Ancien fit RSS : ajustements acceptés à TE EPI",int(fit["target_reliable_mask"].sum()))])
table(list(support_info.items()),("Diagnostic support","Valeur"))

raw_first_rss=np.sqrt(np.sum(np.abs(raw["coil_images"][0])**2,axis=0))
accepted_rates=(np.where(fit["multi_echo"]["support_mask"],fit["multi_echo"]["residual_R2_sInv"],np.nan)
                if complex_echo_enabled else np.where(fit["target_reliable_mask"],fit["R2star_sInv"],np.nan))
fig,axes=plt.subplots(2,3,figsize=(15,9),layout="constrained")
images=[raw_first_rss,fit["support_debiased_amplitude"],fit["object_support"],
        fit["noise_mask"],inputs["field"]["b0Hz"],accepted_rates]
amplitude_title=("Amplitude après retrait du bruit RSS" if support_info["amplitude_debiased"]
                 else "RSS brut : bruit non calibré")
titles=["Premier écho RSS brut",amplitude_title,"Support objet fondé sur le SNR",
        "Masque bruit indépendant","B0 : échos précoces [Hz]",("Décroissance résiduelle complexe [s⁻¹]" if complex_echo_enabled else "R2* local accepté à TE EPI [s⁻¹]")]
for ax,image,title in zip(axes.ravel(),images,titles):
    im=image_physical(ax,image,raw["X_m"],raw["Y_m"],title,
                      cmap="RdBu_r" if "[Hz]" in title else "gray")
    fig.colorbar(im,ax=ax,shrink=.8)
show(fig)

### Mesurer les courants : fichier `waveform_adc.csv`

`WAVEFORM_ADC_INTERVAL_S=0.000050` règle le pas entre deux mesures (50 µs).
Le CSV contient une seule colonne numérique, sans en-tête : `0`, `0.000050`,
`0.000100`, etc. Le compteur repart de zéro à chaque trigger `ext1` vérifié
à partir de la séquence, y compris lorsqu’elle contient un trigger par ligne.

Par défaut, la fenêtre va du premier trigger à la fin du dernier readout image.
`WAVEFORM_ADC_DURATION_S` permet de fixer une autre durée totale depuis le
premier trigger, en secondes. Le prochain trigger ou la fin de la fenêtre est
exclu du segment précédent. Le JSON associé conserve les temps absolus des
triggers et les indices des blocs d’échantillons.

Le fichier est enregistré dans `INPUT_DIR/waveform`, indépendamment de
l’optimisation image et des shims de champ. Une relance réutilise un fichier
identique ou ajoute un numéro si ses paramètres changent.


In [ ]:
adc_waveform_export=None
if EXPORT_ADC_WAVEFORM:
    from adc_waveform_export import export_adc_waveform
    adc_waveform_export=export_adc_waveform(
        inputs["paths"]["scan_sequence"],inputs["timing"],WAVEFORM_OUTPUT_DIR,
        interval_s=WAVEFORM_ADC_INTERVAL_S,trigger_channel=TRIGGER_CHANNEL,
        duration_s=WAVEFORM_ADC_DURATION_S)
    print("CSV ADC :",adc_waveform_export["csv_file"])
    print("Pas ADC [s] :",adc_waveform_export["interval_s"],
          "| échantillons :",adc_waveform_export["samples"],
          "| triggers :",adc_waveform_export["trigger_count"])
else:
    print("Export ADC désactivé.")


In [ ]:
# Diagnostic de l’ancien fit RSS global ; le modèle complexe spatial est vérifié ci-dessous.
times=np.asarray(inputs["echo_times_s"])
mean_echoes=np.asarray(fit["pooled"].get("mean_RSS_echoes",[]))
fig,ax=plt.subplots(figsize=(10,4),layout="constrained")
if len(mean_echoes)==len(times):
    ax.plot(times*1000,mean_echoes,"o",label="Moyenne des amplitudes dé-biaisées dans la ROI globale")
    rate=fit["pooled"].get("R2star_sInv",np.nan)
    anchor_amplitude=fit["pooled"].get("fitted_amplitude_at_reference",np.nan)
    if np.isfinite(rate) and np.isfinite(anchor_amplitude):
        prediction_time=np.linspace(times[0],max(times[-1],sequence_assessment["EPI_ADC_range_s"][1]),300)
        prediction=anchor_amplitude*np.exp(-rate*(prediction_time-times[0]))
        ax.plot(prediction_time*1000,prediction,"--",label="Fit global (acceptation indiquée dans le tableau)")
ax.axvspan(times[0]*1000,times[-1]*1000,color="tab:green",alpha=.1,label="Plage mesurée multi-échos")
ax.axvline(acquisition_native["TE_s"]*1000,color="tab:red",label="TE EPI")
ax.axvline(sequence_assessment["EPI_ADC_range_s"][1]*1000,color="tab:gray",linestyle=":",label="Dernier ADC EPI")
ax.set(xlabel="Temps depuis RF [ms]",ylabel="Amplitude effective",title="Diagnostic RSS global : couverture temporelle")
ax.legend(fontsize=8);show(fig)
if not complex_echo_enabled and not fit["target_reliable_mask"].any():
    print("Aucun ajustement local n'est accepté à TE EPI : vérifier le SNR tardif et les diagnostics avant d'interpréter la correction.")

### 2 bis — Vérifier les trous et la prédiction des échos tardifs

Chaque colonne compare une même image MEGRE mesurée, l’ancienne prédiction avec
replis, la nouvelle prédiction complexe, puis son erreur, à échelle commune.
Le masque de trous est descriptif : pixels brillants au premier écho, devenus
inférieurs à 25 % à l’écho 15. Il ne modifie ni le support ni le coût du fit.
La comparaison après réajustement des 32 échos est distincte des erreurs sur les
échos réservés, enregistrées dans le rapport de sélection.
Les quatre lignes utilisent le même support B0 mesuré ; les pixels non couverts sont masqués.


In [ ]:
if complex_echo_enabled:
    ce=fit["multi_echo"]
    echo_indices=[0,7,14,19,25,31]
    measured=fit["debiased_RSS"]
    predicted=ce["predicted_RSS"]
    previous=complex_echo_diagnostics["legacy_predicted_RSS"]
    vmax=float(np.nanpercentile(measured[0][ce["support_mask"]],99))
    fig,axes=plt.subplots(4,len(echo_indices),figsize=(21,13),layout="constrained")
    for column,index in enumerate(echo_indices):
        for row,(image,label) in enumerate([(measured[index],"Mesuré"),
            (previous[index],"Ancien modèle"),(predicted[index],"Fit complexe"),
            (predicted[index]-measured[index],"Erreur du fit")]):
            signed=row==3
            axes[row,column].set_facecolor("black")
            image_physical(axes[row,column],np.where(ce["support_mask"],image,np.nan),raw["X_m"],raw["Y_m"],
                f"{label} : écho {index+1} / {inputs['echo_times_s'][index]*1000:.1f} ms",
                cmap="RdBu_r" if signed else "gray",vmin=-vmax if signed else 0,vmax=vmax)
    show(fig)
    regions=[("Support mesuré",ce["support_mask"]),
             ("Trous à l’écho 15",complex_echo_diagnostics["hole_mask"])]
    fig,axes=plt.subplots(1,2,figsize=(13,4),layout="constrained")
    for ax,(label,region) in zip(axes,regions):
        if not region.any():
            ax.set_title(label+" : aucun pixel");continue
        scale=np.maximum(fit["first_echo_amplitude"][region],np.finfo(float).tiny)
        for data,name,style in [(measured,"Mesuré","o"),(previous,"Ancien modèle","--"),
                                (predicted,"Fit complexe","-")]:
            ax.plot(inputs["echo_times_s"]*1000,np.median(data[:,region]/scale[None],axis=1),
                    style,label=name)
        ax.axvspan(inputs["echo_times_s"][-COMPLEX_VALIDATION_ECHOES]*1000,
                   inputs["echo_times_s"][-1]*1000,color="tab:green",alpha=.1,
                   label="Échos réservés avant refit")
        ax.set(title=label,xlabel="TE [ms]",ylabel="Amplitude / premier écho")
        ax.legend(fontsize=8)
    show(fig)

## 3. Appliquer rotation et miroirs, puis échantillonner les bobines 3D

Les valeurs d'amplitude et de B0 restent attachées aux mêmes positions natives.
Q transforme leur repère d'imagerie vers les bobines. P transforme ensuite le repère
commun des spins, du k-space et de la reconstruction. Les ADC ne sont ni interpolés,
ni réordonnés ni supprimés.

Chaque pixel natif possède huit positions suivant l'épaisseur. Les réponses ACDC sont
interpolées dans le volume 3D à chacune de ces positions. Un parent incomplet est exclu
sans renormalisation ; la fraction de signal perdue est affichée. Le domaine simulé
est indépendant d'une ROI d'optimisation. Les contraintes de champ portent par défaut
sur **tous les sous-spins conservés**.

In [ ]:
# Les acquisitions synthétiques ont été générées avec des spins ponctuels.
# Les données réelles utilisent par défaut le modèle qui intègre les voxels.
manifest_format=(inputs.get("manifest") or {}).get("format")
is_point_synthetic=manifest_format in {"mrzero_synthetic_acquisition","mrzero_synthetic_twix"}
if SIMULATION_MODEL not in {"auto","point","finite_voxel"}:
    raise ValueError("SIMULATION_MODEL doit valoir auto, point ou finite_voxel.")
if B0_WEIGHTING not in {"auto","uniform","echo_power"}:
    raise ValueError("B0_WEIGHTING doit valoir auto, uniform ou echo_power.")
effective_simulation_model=(("point" if is_point_synthetic else "finite_voxel")
                            if SIMULATION_MODEL=="auto" else SIMULATION_MODEL)
effective_b0_weighting=(("uniform" if is_point_synthetic else "echo_power")
                        if B0_WEIGHTING=="auto" else B0_WEIGHTING)
if complex_echo_enabled and effective_simulation_model!="finite_voxel":
    raise ValueError("Le fit complexe avec gradients connus demande SIMULATION_MODEL=auto ou finite_voxel.")
print("Modèle EPI :",effective_simulation_model,"| pondération B0 :",effective_b0_weighting)
prepared=prepare_echo_shim_inputs(inputs,fit,CALIBRATION_PATH,CALIBRATION_PARAMETERS,
    **angles,subspins_z=SUBSPINS_Z,rate_mode=effective_rate_mode,b0_mode=B0_MODE,
    b0_sigma_pixels=B0_SIGMA_PIXELS,current_response=CURRENT_RESPONSE,
    b0_weighting=effective_b0_weighting,inplane_voxel=effective_simulation_model=="finite_voxel")
experiment=prepared["expanded_experiment"]
calibration=prepared["expanded_calibration"]
preparation_report=prepared["report"]
assert np.allclose(prepared["combined_frame_matrix"],O,rtol=0,atol=1e-12)
assert preparation_report["all_children_in_field_limits"]
table([(key,preparation_report[key]) for key in ["requested_parent_count","baseline_B0_parent_count",
    "retained_parent_count","subspin_count","excluded_missing_B0_amplitude_fraction",
    "excluded_ACDC_amplitude_fraction_of_baseline","amplitude_area_ratio","slice_thickness_mm"]])
print("Miroir / rotation : aucune interpolation des tableaux d'image.")
if effective_simulation_model=="finite_voxel":
    table([(key,preparation_report[key]) for key in ["inplane_voxel_enabled",
        "excluded_missing_B0_gradient_parent_count","excluded_missing_ACDC_gradient_or_corner_parent_count",
        "inplane_voxel_field_limit_corners_per_child","ACDC_corner_linearization_max_abs_error_Hz_per_A"]],
        ("Intégration dans le plan","Valeur"))

fig,axes=plt.subplots(1,3,figsize=(15,4),layout="constrained")
image_physical(axes[0],prepared["native_amplitude"],prepared["physical_X_m"],prepared["physical_Y_m"],
               "Objet dans le repère des bobines")
image_physical(axes[1],prepared["native_B0_Hz"],prepared["physical_X_m"],prepared["physical_Y_m"],
               "B0 dans le repère des bobines [Hz]",cmap="RdBu_r")
image_physical(axes[2],prepared["parent_valid_mask"],prepared["physical_X_m"],prepared["physical_Y_m"],
               "Couverture des huit sous-spins")
for column,color,label in [(0,"tab:red","+X données"),(1,"tab:blue","+Y données")]:
    axes[0].arrow(0,0,25*Q[0,column],25*Q[1,column],color=color,width=.8,length_includes_head=True)
    axes[0].text(30*Q[0,column],30*Q[1,column],label,color=color,fontsize=8)
show(fig)
# Instantané original pour les relances de construction et de calibration.
uncalibrated_prepared=deepcopy(prepared)

## 4. Construire MRzero avec les paramètres physiques du 04

L'amplitude est ancrée au premier TE MEGRE mesuré, puis évolue jusqu'aux ADC EPI.
Les poids d'aire entre la grille native et la grille EPI, ainsi que les fractions de volume suivant z, sont appliqués une seule fois.
Le B0 suivant z reste inconnu ; il est constant dans la coupe, tandis que les cartes ACDC
varient réellement avec z. Le diagnostic B0 nul conserve cet objet et cette relaxation ; il ne sert pas de cible d’optimisation.

La réponse matérielle et son intégrale de phase restent continues entre les lignes.
Le solveur utilise l'adjoint exact de la reconstruction dans le repère choisi.

En mode complexe, l’amplitude transmise au phantom est l’enveloppe au premier TE,
sans les facteurs de déphasage explicites. La dispersion non résolue est appliquée
au signal transverse natif, une seule fois, pour chaque sous-spin. Elle reste fixe
pendant le shim et dans le diagnostic B0 nul ; aucune répartition B0 suivant z n’est déduite
d’une coupe unique. Les gradients ACDC et leurs dérivées restent ceux du modèle
intégré dans le plan.

In [ ]:
def reconstruction_factory(acquisition,device,dtype):
    return FrameReconstruction(acquisition,device,dtype,frame_matrix=O,
                               sample_chunk=RECONSTRUCTION_CHUNK)

# Repartir de la préparation non calibrée même si cette cellule est relancée seule.
prepared=deepcopy(uncalibrated_prepared)
experiment=prepared["expanded_experiment"]
calibration=prepared["expanded_calibration"]
preparation_report=prepared["report"]
if "model" in globals():
    import gc
    del model
    gc.collect()
    if torch.cuda.is_available():torch.cuda.empty_cache()

model_class=(MultiEchoVoxelACDCModel if complex_echo_enabled else
             FiniteVoxelEchoACDCModel if effective_simulation_model=="finite_voxel" else EchoSubspinACDCModel)
model=model_class(experiment,calibration,max_current_A=CHANNEL_MAX_CURRENT_A,
    hardware=HARDWARE,reconstruction_factory=reconstruction_factory,
    profile=GPU_PROFILE,precision=GPU_PRECISION,memory_fraction=GPU_MEMORY_FRACTION,
    sample_chunk=SAMPLE_CHUNK,require_cuda=GPU_REQUIRED)
table([("Périphérique / précision",f"{model.device} / {model.runtime_info['precision']}"),
       ("PyTorch / runtime CUDA",f"{torch.__version__} / {torch.version.cuda}"),
       ("Carte détectée",model.runtime_info["detected_gpu_name"])])
X_image,Y_image=model.reconstruction.X_m,model.reconstruction.Y_m
measured_coils=model.reconstruction.numpy_image(
    np.moveaxis(inputs["scan_diagnostics"]["corrected_kspace"],1,0))
measured_image=np.sqrt(np.sum(abs(measured_coils)**2,axis=0))
zero_currents=np.zeros((model.nc,model.nstates))
ideal_image=model.ideal_image
baseline_image,baseline_kdata=model.forward(zero_currents)
baseline_model_mse=float(np.sum((abs(baseline_image)-abs(ideal_image))**2)/np.sum(abs(ideal_image)**2))
baseline_measured_metrics=compare_magnitude(measured_image,baseline_image,X_m=X_image,Y_m=Y_image)
table([("Spins simulés",int(model.support.sum())),("Stockage du cache MRzero",model.runtime_info["native_baseline_cache"]),
       ("MSE diagnostique sans ACDC / simulation B0 nul",baseline_model_mse),
       ("NRMSE du twin / EPI réelle [%]",baseline_measured_metrics["NRMSE_percent"])])
# Comparaison à l'ancien modèle sur exactement les mêmes parents conservés.
# Comparaison AVANT l'étape de calibration de la baseline EPI.
point_baseline_image=None
point_baseline_metrics=None
fidelity_comparison=None
if COMPARE_POINT_BASELINE and effective_simulation_model=="finite_voxel":
    legacy_b0=np.asarray(inputs["field"]["b0Hz"])
    if B0_MODE=="regularized":
        legacy_b0=regularize_b0_local_plane(legacy_b0,inputs["field"]["mask"],
                                           sigma_pixels=B0_SIGMA_PIXELS)["b0Hz"]
    # Ancien objet RSS, pas la nouvelle enveloppe privée de ses facteurs de cohérence.
    point_amplitude=prepared["native_amplitude"]
    point_rate=prepared["native_rate_sInv"]
    if complex_echo_enabled:
        if RATE_MODE=="guarded":
            point_amplitude=legacy_echo_fit["simulation_amplitude_at_reference"]
            point_rate=legacy_echo_fit["simulation_R2star_sInv"]
        else:
            point_amplitude=legacy_echo_fit["first_echo_amplitude"]
            pooled_rate=(legacy_echo_fit["pooled"].get("R2star_sInv",0.)
                         if RATE_MODE=="pooled" and legacy_echo_fit["pooled"].get("target_reliable_mask",False) else 0.)
            point_rate=np.full(point_amplitude.shape,pooled_rate)
    point_check=NativeEPITwin(experiment["acquisition"],prepared["frame_X_m"],prepared["frame_Y_m"],
        point_amplitude*preparation_report["amplitude_area_ratio"],legacy_b0,
        prepared["parent_valid_mask"],rate_sInv=point_rate,
        amplitude_reference_time_s=fit["amplitude_reference_time_s"],device=str(model.device),sample_chunk=256)
    point_result=point_check.simulate(b0_enabled=True)
    point_baseline_image=model.reconstruction.numpy_image(point_result["kdata"][0])
    point_baseline_metrics=compare_magnitude(measured_image,point_baseline_image,X_m=X_image,Y_m=Y_image)
    bright_roi=measured_image>.2*measured_image.max()
    def low_signal_count(image,metrics,fraction):
        return int(np.count_nonzero(bright_roi & (metrics["gain"]*np.abs(image)<fraction*measured_image)))
    fidelity_comparison={"comparison":"same retained native parents; full-FOV evaluation gain only",
        "legacy_counter_direction":"simulation lower than measured EPI only; measured-dark regions excluded",
        "bright_ROI_fraction_of_measured_peak":.2,"bright_ROI_pixels":int(bright_roi.sum()),
        "point_NRMSE_percent":point_baseline_metrics["NRMSE_percent"],
        "revised_NRMSE_percent":baseline_measured_metrics["NRMSE_percent"],
        "point_below_half_measured_pixels":low_signal_count(point_baseline_image,point_baseline_metrics,.5),
        "revised_below_half_measured_pixels":low_signal_count(baseline_image,baseline_measured_metrics,.5)}
    fidelity_labels={"point_below_half_measured_pixels":"Ponctuel : simulation < 50 % du mesuré clair [pixels]",
        "revised_below_half_measured_pixels":"Voxels finis : simulation < 50 % du mesuré clair [pixels]"}
    table([(fidelity_labels.get(key,key),value) for key,value in fidelity_comparison.items()],
          ("Ancien compteur : déficits simulés uniquement","Valeur"))
    panels=[baseline_measured_metrics["gain"]*np.abs(ideal_image),
            point_baseline_metrics["gain"]*np.abs(point_baseline_image),
            baseline_measured_metrics["gain"]*np.abs(baseline_image),measured_image]
    titles=["Diagnostic EPI : B0 nul","Ponctuel : objet RSS historique",("Voxels finis + cohérence 32 échos" if complex_echo_enabled else "Voxels finis avant calibration"),"EPI réelle sans ACDC"]
    del point_check,point_result
else:
    gain=baseline_measured_metrics["gain"]
    panels=[gain*np.abs(ideal_image),gain*np.abs(baseline_image),measured_image]
    titles=["MRzero : B0 nul","MRzero : B0 mesuré","EPI sans ACDC (données d'entrée)"]
fig,axes=plt.subplots(1,len(panels),figsize=(5*len(panels),4),layout="constrained")
for ax,image,title in zip(axes,panels,titles):
    image_physical(ax,image,X_image,Y_image,title,vmin=0,vmax=measured_image.max()*1.1)
show(fig)
# Archives immuables : la calibration ne repart jamais de son propre résultat.
uncalibrated_baseline_image=baseline_image.copy()
uncalibrated_ideal_image=ideal_image.copy()
uncalibrated_baseline_kdata=baseline_kdata.copy()
uncalibrated_sample_chunk=int(model.sample_chunk)  # Partition native conservée lors des reconstructions.
uncalibrated_baseline_model_mse=baseline_model_mse
uncalibrated_baseline_measured_metrics=deepcopy(baseline_measured_metrics)

# Toute reconstruction invalide le diagnostic précédent.
baseline_signal_disagreement=None
baseline_signal_disagreement_arrays={}

### 4 bis — Calibrer les petits écarts de l’EPI sans ACDC

`BASELINE_CALIBRATION="auto"` active cette étape pour les acquisitions mesurées ; les jeux
synthétiques restent inchangés. `"off"` la désactive et `"on"` la demande explicitement.
Les bornes sont fixées avant l’ajustement : décalage fréquentiel global ±30 Hz,
translation suivant la lecture ±2 mm et phase résiduelle impair/pair constante ±0,15 rad.
La translation suivant la phase et la rotation restent fixes, car elles se confondent
en partie avec le décalage fréquentiel ou l’alignement des bobines.

L’objet, la relaxation et les variations spatiales de B0 restent fixés. Des blocs spatiaux
distincts servent à l’ajustement, à la sélection et au contrôle final ; les gains d’affichage
sont estimés uniquement sur les pixels d’ajustement. Il s’agit d’une **validation interne
d’une seule acquisition**, déjà utilisée pour développer le modèle, et non d’un scan indépendant.
Un petit gain peut être insuffisant pour expliquer les artefacts restants.

L’application est refusée si la sélection ne s’améliore pas, si une borne est atteinte,
si les paramètres sont mal identifiés, ou si le contrôle final ne s’améliore pas.
Les paramètres retenus reconstruisent le modèle natif MRzero : translation physique avant
rééchantillonnage des cartes ACDC, offset B0 et phase ADC résiduelle après la correction des
navigateurs. Le diagnostic B0 nul retire l’offset fréquentiel et conserve la géométrie et la phase
matérielle calibrées. Cette étape ne modifie aucun courant ni aucune donnée mesurée.

Les figures ponctuel/voxels précédentes décrivent le modèle **avant calibration**. Relancer
cette cellule repart des données originales, y compris après un refus ou une désactivation.
Après toute modification de cette étape, relancer ensuite la vérification et l’optimisation.


In [ ]:
# Une relance repart des mêmes échantillons non calibrés, sans cumuler les paramètres.
import gc
import importlib
import baseline_calibration as baseline_calibration_tools
import baseline_metadata_audit as baseline_audit_tools
baseline_calibration_tools=importlib.reload(baseline_calibration_tools)
baseline_audit_tools=importlib.reload(baseline_audit_tools)
if BASELINE_CALIBRATION not in {"auto","off","on"}:
    raise ValueError("BASELINE_CALIBRATION doit valoir auto, off ou on.")
baseline_calibration_enabled=(BASELINE_CALIBRATION=="on" or
    (BASELINE_CALIBRATION=="auto" and not is_point_synthetic))
baseline_metadata_audit=baseline_audit_tools.audit_baseline_inputs(inputs)
baseline_calibration_result=None
baseline_calibration_masks={name:np.zeros(measured_image.shape,dtype=bool)
                            for name in ("fit_mask","validation_mask","test_mask")}
baseline_calibration_parameters={name:0. for name in baseline_calibration_tools.PARAMETER_NAMES}
if baseline_calibration_enabled:
    baseline_calibration_result=baseline_calibration_tools.fit_baseline_calibration(
        inputs["acquisition"],uncalibrated_baseline_kdata,measured_image,bounds=CALIBRATION_BOUNDS)
    baseline_calibration_report=deepcopy(baseline_calibration_result["report"])
    baseline_calibration_parameters=baseline_calibration_result["parameters"].copy()
    for name in baseline_calibration_masks:
        baseline_calibration_masks[name]=baseline_calibration_result[name].copy()
else:
    baseline_calibration_report={"accepted":False,"parameters":baseline_calibration_parameters.copy(),
        "reason":"Calibration désactivée pour ce jeu synthétique." if is_point_synthetic
                 else "Calibration désactivée par BASELINE_CALIBRATION=off.",
        "gain_applied_to_object":False,"independent_acquisition_validation":False}
baseline_calibration_report["requested_mode"]=BASELINE_CALIBRATION
baseline_calibration_report["source_sha256"]=dict(inputs["source_sha256"],
    calibration=uncalibrated_prepared["report"]["calibration_source_sha256"])

def rebuild_baseline_model(parameters):
    rebuilt=prepare_echo_shim_inputs(inputs,fit,CALIBRATION_PATH,CALIBRATION_PARAMETERS,
        **angles,subspins_z=SUBSPINS_Z,rate_mode=effective_rate_mode,b0_mode=B0_MODE,
        b0_sigma_pixels=B0_SIGMA_PIXELS,current_response=CURRENT_RESPONSE,
        b0_weighting=effective_b0_weighting,inplane_voxel=effective_simulation_model=="finite_voxel",
        baseline_calibration=parameters)
    rebuilt_model=model_class(rebuilt["expanded_experiment"],rebuilt["expanded_calibration"],
        max_current_A=CHANNEL_MAX_CURRENT_A,hardware=HARDWARE,reconstruction_factory=reconstruction_factory,
        profile=GPU_PROFILE,precision=GPU_PRECISION,memory_fraction=GPU_MEMORY_FRACTION,
        sample_chunk=uncalibrated_sample_chunk,require_cuda=GPU_REQUIRED)
    return rebuilt,rebuilt_model

# Libérer le cache natif de l'ancien modèle avant de construire le suivant.
del model
gc.collect()
if torch.cuda.is_available():torch.cuda.empty_cache()
prepared,model=rebuild_baseline_model(baseline_calibration_parameters)
zero_currents=np.zeros((model.nc,model.nstates))
baseline_image,baseline_kdata=model.forward(zero_currents)
expected_calibrated_kdata=(uncalibrated_baseline_kdata*
    baseline_calibration_tools.phase_factor(inputs["acquisition"],baseline_calibration_parameters))
baseline_calibration_native_phase_parity=float(np.linalg.norm(baseline_kdata-expected_calibrated_kdata)
    /max(np.linalg.norm(expected_calibrated_kdata),np.finfo(float).tiny))
def fixed_object_rebuild_checks(candidate):
    checks={key:bool(np.array_equal(candidate[key],uncalibrated_prepared[key],equal_nan=True))
        for key in ("parent_flat_index","parent_valid_mask","weights","native_amplitude","native_rate_sInv")}
    checks["subspin_amplitude"]=bool(np.array_equal(candidate["expanded_experiment"]["m0_proxy"],
        uncalibrated_prepared["expanded_experiment"]["m0_proxy"],equal_nan=True))
    return checks
baseline_fixed_object_checks=fixed_object_rebuild_checks(prepared)
baseline_calibration_report["fixed_object_rebuild_checks"]=baseline_fixed_object_checks.copy()
if baseline_calibration_native_phase_parity>=1e-4 or not all(baseline_fixed_object_checks.values()):
    baseline_calibration_report["phase_parity_application_rejected"]=True
    baseline_calibration_report["candidate_native_phase_parity"]=baseline_calibration_native_phase_parity
    baseline_calibration_report["parameters_before_rebuild_gate"]=baseline_calibration_parameters.copy()
    baseline_calibration_report["accepted"]=False
    baseline_calibration_report["reason"]="Parité native insuffisante ou support/amplitude/relaxation/poids modifiés après rééchantillonnage ; paramètres nuls restaurés."
    baseline_calibration_parameters={name:0. for name in baseline_calibration_tools.PARAMETER_NAMES}
    baseline_calibration_report["parameters"]=baseline_calibration_parameters.copy()
    if "baseline" in baseline_calibration_report:
        baseline_calibration_report["applied"]=deepcopy(baseline_calibration_report["baseline"])
    del model
    gc.collect()
    if torch.cuda.is_available():torch.cuda.empty_cache()
    prepared,model=rebuild_baseline_model(baseline_calibration_parameters)
    zero_currents=np.zeros((model.nc,model.nstates))
    baseline_image,baseline_kdata=model.forward(zero_currents)
    baseline_calibration_native_phase_parity=float(np.linalg.norm(baseline_kdata-uncalibrated_baseline_kdata)
        /max(np.linalg.norm(uncalibrated_baseline_kdata),np.finfo(float).tiny))
baseline_calibration_report["applied_fixed_object_rebuild_checks"]=fixed_object_rebuild_checks(prepared)
assert (baseline_calibration_native_phase_parity<1e-4 and
        all(baseline_calibration_report["applied_fixed_object_rebuild_checks"].values())),"Le modèle restauré ne correspond plus à la baseline originale ; relancer les entrées."
experiment=prepared["expanded_experiment"]
calibration=prepared["expanded_calibration"]
preparation_report=prepared["report"]
X_image,Y_image=model.reconstruction.X_m,model.reconstruction.Y_m
ideal_image=model.ideal_image
baseline_model_mse=float(np.sum((abs(baseline_image)-abs(ideal_image))**2)/np.sum(abs(ideal_image)**2))
baseline_measured_metrics=compare_magnitude(measured_image,baseline_image,X_m=X_image,Y_m=Y_image)
baseline_calibration_report["native_phase_parity"]=baseline_calibration_native_phase_parity
baseline_calibration_report["native_sample_chunk"]={"uncalibrated":uncalibrated_sample_chunk,
    "requested_rebuild":uncalibrated_sample_chunk,"effective_rebuild":int(model.sample_chunk)}
baseline_calibration_report["preparation"]=deepcopy(preparation_report)
model.runtime_info["baseline_calibration"]=baseline_calibration_parameters.copy()
model.runtime_info["baseline_calibration_accepted"]=baseline_calibration_report["accepted"]
model.runtime_info["baseline_calibration_native_phase_parity"]=baseline_calibration_native_phase_parity
model.runtime_info["baseline_calibration_original_sample_chunk"]=uncalibrated_sample_chunk
# Invalider une optimisation antérieure si cette cellule est relancée seule.
solution=None
image_history=None
solver_ok=False
print(baseline_calibration_report["reason"])
table(list(baseline_calibration_parameters.items()),("Paramètre du modèle","Valeur appliquée"))
if "baseline" in baseline_calibration_report:
    table([(name,baseline_calibration_report["baseline"][name]["NRMSE_percent"],
            baseline_calibration_report["applied"][name]["NRMSE_percent"])
           for name in ("fit","validation","test","full_FOV")],
          ("Pixels / gain ajusté sur fit uniquement","NRMSE avant [%]","NRMSE après [%]"))
    before_gain=baseline_calibration_report["baseline_gain_from_fit_pixels"]
    after_gain=(baseline_calibration_report["candidate_gain_from_fit_pixels"]
                if baseline_calibration_report["accepted"] else before_gain)
else:
    before_gain=uncalibrated_baseline_measured_metrics["gain"]
    after_gain=before_gain
table([("Parité native MRzero / phase rapide",baseline_calibration_native_phase_parity),
       ("MSE diagnostique baseline / simulation B0 nul",baseline_model_mse)])
print("Validation interne d'une seule acquisition : confirmer les paramètres sur un autre scan avant de conclure sur leur origine physique.")
fig,axes=plt.subplots(1,3,figsize=(15,4),layout="constrained")
for ax,image,title in zip(axes,[measured_image,before_gain*np.abs(uncalibrated_baseline_image),
                               after_gain*np.abs(baseline_image)],
                         ["EPI mesurée sans ACDC","MRzero avant calibration","MRzero après calibration"]):
    image_physical(ax,image,X_image,Y_image,title,vmin=0,vmax=measured_image.max()*1.1)
show(fig)
OUTPUT_DIR.mkdir(parents=True,exist_ok=True)
np.savez_compressed(OUTPUT_DIR/"baseline_calibration.npz",
    uncalibrated_baseline_image=uncalibrated_baseline_image,
    uncalibrated_ideal_image=uncalibrated_ideal_image,
    uncalibrated_baseline_kdata=uncalibrated_baseline_kdata,
    calibrated_baseline_image=baseline_image,calibrated_ideal_image=ideal_image,
    calibrated_baseline_kdata=baseline_kdata,measured_image=measured_image,
    **baseline_calibration_masks)
(OUTPUT_DIR/"baseline_calibration.json").write_text(
    json.dumps(json_safe(baseline_calibration_report),indent=2,ensure_ascii=False,allow_nan=False),encoding="utf-8")
(OUTPUT_DIR/"baseline_metadata_audit.json").write_text(
    json.dumps(json_safe(baseline_metadata_audit),indent=2,ensure_ascii=False,allow_nan=False),encoding="utf-8")

# Relancer ensuite le diagnostic des deux sens avec ce modèle.
baseline_signal_disagreement=None
baseline_signal_disagreement_arrays={}

### 4 ter — Vérifier les pertes de signal dans les deux sens

La MSE de calibration compare déjà les intensités **dans les deux sens**, y compris les
pixels sombres du mesuré. En revanche, les anciens compteurs « simulation sous 50 % du
mesuré » ne regardaient que les pixels où l’EPI mesurée était claire. Ils excluaient donc
les grandes zones noires réelles et mesuraient uniquement les déficits propres à la simulation.

Ce diagnostic distingue maintenant les **zones sombres réelles non reproduites** (simulation
trop claire, rouge) des **déficits simulés absents du mesuré** (simulation trop sombre, bleu).
Il rapporte aussi les régions où les deux images sont sous le seuil, sans les considérer
comme un accord : leur différence continue peut rester importante. Les indicateurs de
rapport utilisent deux seuils fixes : 25 % et 50 %, et exigent que l'image la plus claire
dépasse 10 % du maximum mesuré, avec la même échelle pour chaque modèle. Le champ complet est le
diagnostic principal. Une enveloppe commune tirée du diagnostic B0 nul non calibré fournit un second
regard sur l’intérieur de l’objet ; son remplissage concerne exclusivement ce masque de
diagnostic, sans toucher au support des spins, à la GRE mesurée, au B0 ou à la fonction de coût.

Les gains de comparaison sont ceux de la calibration, estimés sur les pixels d’ajustement
uniquement lorsqu’elle est active ; sinon le gain d’évaluation original est conservé. La même
enveloppe et la même échelle s’appliquent aux modèles comparés. Les blocs de validation/test
restent séparés. Ces désaccords d’intensité ne prouvent pas une origine physique particulière,
un gradient suivant z ou une perte irréversible de signal. L’optimiseur ACDC compare ensuite son image à la GRE mesurée. Un défaut du modèle
peut donc limiter la correction prédite ; ces diagnostics évaluent sa fidélité à l’EPI
sans ACDC avant toute optimisation des courants.


In [ ]:
# Diagnostic seulement : aucun changement d'objet, de champ ou de fonction de coût.
import signal_disagreement as signal_disagreement_tools
signal_disagreement_tools=importlib.reload(signal_disagreement_tools)
signal_reference_scale=float(measured_image.max())
signal_envelope=signal_disagreement_tools.fixed_comparison_envelope(
    np.abs(uncalibrated_ideal_image),threshold_fraction=.1)
signal_subsets={name.removesuffix("_mask"):mask for name,mask in baseline_calibration_masks.items()
                if np.any(mask)}
signal_sources={"before_calibration":(uncalibrated_baseline_image,before_gain),
                "after_calibration":(baseline_image,after_gain)}
if point_baseline_image is not None:
    signal_sources={"point_before_calibration":(point_baseline_image,before_gain),**signal_sources}
baseline_signal_disagreement={"comparison_policy":{
    "primary_domain":"full FOV, including measured-dark pixels",
    "secondary_domain":"fixed envelope from uncalibrated zero-B0 target; diagnostic only",
    "envelope_threshold_fraction":.1,"reference_scale":signal_reference_scale,
    "bright_fraction":.1,"both_low_does_not_imply_agreement":True,
    "gain_source":"calibration fit pixels only" if baseline_calibration_enabled else "original full-FOV evaluation gain",
    "point_comparison_gain":"same before-calibration gain as the uncalibrated finite-voxel model",
    "object_target_field_and_optimization_loss_unchanged":True,
    "independent_physical_validation":False},"models":{}}
baseline_signal_disagreement_arrays={"fixed_comparison_envelope_mask":signal_envelope.copy(),"X_image_m":X_image.copy(),"Y_image_m":Y_image.copy()}
signal_rows=[]
signal_energy_rows=[]
signal_after_full=None
for name,(simulated,comparison_gain) in signal_sources.items():
    baseline_signal_disagreement["models"][name]={}
    for domain,analysis_mask in (("full_FOV",None),("fixed_envelope",signal_envelope)):
        result=signal_disagreement_tools.signal_disagreement(measured_image,simulated,
            gain=comparison_gain,analysis_mask=analysis_mask,subsets=signal_subsets or None,
            reference_scale=signal_reference_scale,bright_fraction=.1,ratios=(.25,.5))
        baseline_signal_disagreement["models"][name][domain]=result["report"]
        for key,value in result["arrays"].items():
            baseline_signal_disagreement_arrays[name+"_"+domain+"_"+key]=value
        maps=result["arrays"]
        energy=result["report"]["analysis_mask"]
        signal_energy_rows.append((name,domain,energy["NRMSE_percent"],
            energy["overprediction_normalized_SSE"],energy["underprediction_normalized_SSE"],
            energy["both_low_fraction_of_SSE"]))
        signal_rows.append((name,domain,
            int(np.count_nonzero(maps["measured_low_simulated_high_50_mask"])),
            int(np.count_nonzero(maps["simulated_low_measured_high_50_mask"])),
            int(np.count_nonzero(maps["both_low_mask"]))))
        if name=="after_calibration" and domain=="full_FOV":signal_after_full=result
table(signal_energy_rows,("Modèle","Domaine","NRMSE [%]","SSE normalisée : simulation trop claire",
    "SSE normalisée : simulation trop sombre","Fraction SSE : deux sous seuil"))
table(signal_rows,("Modèle","Domaine","Réel sombre / simulation trop claire [pixels]",
    "Simulation sombre / réel clair [pixels]","Deux sous seuil, accord non garanti [pixels]"))
print("Les compteurs anciens ne mesuraient que la simulation trop sombre. Les pertes de signal réelles insuffisamment reproduites doivent aussi être examinées ; les indicateurs colorés ne prouvent pas leur origine physique.")
print("SSE signée continue prioritaire ; indicateurs descriptifs à rapports 25 % / 50 %, image claire >= 10 % du maximum mesuré fixe. Deux images sous seuil ne prouvent pas leur accord.")
maps=signal_after_full["arrays"]
classification=np.full(measured_image.shape,np.nan)
classification[maps["simulated_low_measured_high_50_mask"]]=-1.
classification[maps["measured_low_simulated_high_50_mask"]]=1.
difference=maps["difference"]
diff_max=max(float(np.nanmax(np.abs(difference))),np.finfo(float).tiny)
fig,axes=plt.subplots(1,4,figsize=(20,4),layout="constrained")
image_physical(axes[0],measured_image,X_image,Y_image,"EPI mesurée sans ACDC",vmin=0,vmax=signal_reference_scale)
image_physical(axes[1],after_gain*np.abs(baseline_image),X_image,Y_image,"Baseline MRzero comparée",vmin=0,vmax=signal_reference_scale)
im=image_physical(axes[2],difference,X_image,Y_image,"Simulation − mesuré\nRouge : simulation trop claire",cmap="RdBu_r",vmin=-diff_max,vmax=diff_max)
fig.colorbar(im,ax=axes[2],shrink=.8)
image_physical(axes[3],measured_image,X_image,Y_image,"Rouge : mesure plus faible que simulation\nBleu : simulation plus faible (rapport 50 %, clarté 10 %)",vmin=0,vmax=signal_reference_scale)
axes[3].pcolormesh(X_image*1000,Y_image*1000,classification,shading="auto",cmap="RdBu_r",vmin=-1,vmax=1,alpha=.8)
show(fig)
OUTPUT_DIR.mkdir(parents=True,exist_ok=True)
np.savez_compressed(OUTPUT_DIR/"baseline_signal_disagreement.npz",**baseline_signal_disagreement_arrays)
(OUTPUT_DIR/"baseline_signal_disagreement.json").write_text(
    json.dumps(json_safe(baseline_signal_disagreement),indent=2,ensure_ascii=False,allow_nan=False),encoding="utf-8")
baseline_calibration_report["native_model_signal_disagreement"]=baseline_signal_disagreement
(OUTPUT_DIR/"baseline_calibration.json").write_text(
    json.dumps(json_safe(baseline_calibration_report),indent=2,ensure_ascii=False,allow_nan=False),encoding="utf-8")


### Vérifier le modèle avant optimisation

Pour le modèle ponctuel, la somme des sous-spins à courant nul doit retrouver
la V4 sur les mêmes parents. Pour les voxels finis, la référence ponctuelle n'est
plus le même modèle : on compare le facteur d'intégration analytique à une quadrature
numérique indépendante. Les contrôles de resommation natifs MRzero restent actifs.
Les contraintes de champ du modèle fini couvrent aussi les coins des voxels.

In [ ]:
if effective_simulation_model=="finite_voxel":
    reference_validation_report=model.validate_voxel_quadrature()
    zero_current_parity=None  # La V4 ponctuelle représente un autre modèle physique.
    table(list(reference_validation_report.items()),("Contrôle quadrature des voxels","Valeur"))
    if not reference_validation_report["passed"]:
        raise AssertionError("L'intégration analytique des voxels ne concorde pas avec la quadrature indépendante.")
else:
    reference_check=NativeEPITwin(experiment["acquisition"],prepared["frame_X_m"],prepared["frame_Y_m"],
        prepared["native_amplitude"]*preparation_report["amplitude_area_ratio"],prepared["native_B0_Hz"],
        prepared["parent_valid_mask"],rate_sInv=prepared["native_rate_sInv"],
        amplitude_reference_time_s=fit["amplitude_reference_time_s"],device=str(model.device),sample_chunk=128)
    # Le signal natif est comparé avant reconstruction, indépendante de l'ordre d'affichage.
    reference_result=reference_check.simulate(b0_enabled=True)
    reference_validation_report=reference_result["report"]
    zero_current_parity=float(np.linalg.norm(reference_result["kdata"][0]-baseline_kdata)/np.linalg.norm(baseline_kdata))
    assert np.isfinite(zero_current_parity) and zero_current_parity<1e-4, (
        f"Échec V4 / modèle complet à courant nul : {zero_current_parity:.3g} (seuil 1e-4).")
    table([("Resommation MRzero : erreur relative au signal",reference_validation_report["native_resummation_relative_error"]),
           ("Resommation MRzero : erreur / tolérance (≤1)",reference_validation_report["native_resummation_max_tolerance_ratio"]),
           ("Erreur relative V4 / modèle complet à courant nul",zero_current_parity)])
    del reference_check,reference_result
    if model.device.type=="cuda":torch.cuda.empty_cache()

## 5. Initialisation par shim statique et contrôle matériel

La solution statique fournit un point de départ à l'optimisation d'image. Tous les
sous-spins participent aux limites de champ. Les valeurs électriques non mesurées
restent désactivées ; aucun paramètre R/L ou thermique n'est inventé.

In [ ]:
if IMAGE_INITIALIZATION=="static":
    static_fit=solve_field_gpu(model.maps_numpy[model.quality_mask],
        -experiment["field"]["b0Hz"][model.quality_mask],device=model.device,dtype=model.dtype,
        max_current_A=CHANNEL_MAX_CURRENT_A,total_current_limit_A=TOTAL_CURRENT_LIMIT_A,
        field_limit_Hz=FIELD_LIMIT_HZ,max_iterations=1000,tolerance=1e-7)
    requested_initial=np.repeat(static_fit["currents_A"][:,:1],model.nstates,axis=1)
elif IMAGE_INITIALIZATION=="zero":requested_initial=zero_currents.copy()
else:raise ValueError("IMAGE_INITIALIZATION doit valoir static ou zero.")
initial_currents=model.project_currents(requested_initial,TOTAL_CURRENT_LIMIT_A,FIELD_LIMIT_HZ)
initial_hardware=model.hardware.diagnostics(initial_currents,TOTAL_CURRENT_LIMIT_A,FIELD_LIMIT_HZ)
assert initial_hardware["all_limits_met"]
initial_image,_=model.forward(initial_currents)
table(list(initial_hardware["checks"].items()),("Contrôle initial","Résultat"))
print("Réponse équivalente : tau [µs] =",model.current_response["tau_s"][:3]*1e6)

## 6. Optimisation des courants avec affichage dynamique

La cible est **le scan GRE mesuré sélectionné**. Sa matrice peut différer de celle
de l'EPI : il est interpolé sur la grille physique EPI, sans extrapolation. Le masque
et l'échelle sont fixés à partir de la baseline avant l'optimisation, puis conservés
pour tous les courants, l'affichage et la vérification après quantification DAC.

Le coût image est `J_image(I) = sum_M((g*abs(EPI_MRzero(I)) - GRE_mesurée)**2) / sum_M(GRE_mesurée**2)`.
`g` est le gain positif fixe simulation→EPI sans ACDC→GRE ; `M` est la couverture
commune (ou la ROI GRE explicitement choisie). Les pixels sombres restent pénalisés.
Le modèle évalué utilise B0, les pertes ajustées aux 32 échos et chaque réponse ACDC
à chaque sous-spin avant la somme complexe. La simulation B0 nul est un diagnostic
séparé et ne remplace jamais la GRE cible. L'orientation intervient dans les gradients.
L'affichage dynamique conserve l'ordre des pixels acquis ; les figures physiques
utilisent les coordonnées transformées. Les graphes CUDA du 03 sont désactivés pour
conserver la projection matérielle du 04.

Le curseur **Scan current** permet de revoir les images au fur et à mesure de leur
arrivée. Seules les magnitudes aux itérations `0, LIVE_UPDATE_EVERY, 2*LIVE_UPDATE_EVERY...`
sont conservées ; aucun historique de courants ou de k-space n'est ajouté.
La courbe de coût garde ses valeurs scalaires à chaque itération. Le dernier affichage
live peut donc correspondre à une itération non archivée ; le meilleur résultat reste
celui de la section 7.

La galerie est sauvegardée à chaque mise à jour dans `optimization_images.npz` et
`optimization_images.html`. Le fichier HTML permet aussi de parcourir les images hors
du notebook. Son échelle reste fixe, identique pour toutes les images archivées.

Le tableau ci-dessous distingue le **périphérique de calcul** du **stockage du
cache MRzero**. Un cache en RAM CPU peut alimenter un calcul CUDA par blocs ;
les transferts peuvent alors réduire l'activité GPU observée. `use_cuda_graph=False`
conserve les contrôles matériels et ne désactive pas le calcul CUDA.
Si le modèle affiche `cpu` alors que le GPU est demandé, reconstruire le modèle
après avoir corrigé l'environnement et redémarré le noyau.

Avec `FIELD_GUIDANCE_WEIGHT > 0`, le solveur minimise
`J_total = J_image + FIELD_GUIDANCE_WEIGHT * mean((B0_residuel / FIELD_GUIDANCE_SCALE_HZ)**2)`.
Le résidu est calculé aux positions physiques des sous-spins fiables, en moyennant
le champ ACDC réel entre les centres temporels de lignes EPI successives. Les
intégrales exactes de réponse des amplificateurs sont utilisées. Le B0, sa moyenne
et ses grands écarts restent en Hz : aucune réduction modulo le FOV ni modulo 2π.
La reconstruction Fourier représente déjà le repliement ; ce guide aide à choisir
la correction physique lorsque plusieurs images repliées se ressemblent.

`0.05` est un poids initial à évaluer, pas un réglage validé pour chaque fantôme.
`0` restaure le comportement image seul. La MSE et la NRMSE affichées restent
strictement des métriques image ; les historiques `objective`, `field_penalty` et
`residual_field_rms_Hz` enregistrent séparément le guidage. Le meilleur résultat
est choisi sur le coût total, pas forcément sur la plus petite MSE image rencontrée.
En mode guidé, `TARGET_MSE` est seulement diagnostique : atteindre une MSE faible
ne suffit pas à arrêter le solveur sur une mauvaise branche de repliement.

Le guide suppose une carte B0 correctement déroulée **et correctement référencée
en fréquence absolue**, avec la même fréquence scanner que l'EPI. Il ne dépasse
aucune limite de courant/champ et garde les canaux désactivés à zéro. Il ne garantit
ni une correction parfaite hors de la capacité des bobines ni la récupération
du signal perdu. Relancer depuis les paramètres après changement du poids.


In [ ]:
# Cible mesurée installée APRÈS la calibration/reconstruction du modèle.
optimization_target=configure_measured_gre_target(model,inputs["gre"],
    baseline_image=baseline_image,measured_baseline_image=measured_image,
    frame_matrix=O,threshold_fraction=OPTIMIZATION_REFERENCE_THRESHOLD)
optimization_reference_image=optimization_target["reference_image"]
optimization_reference_mask=optimization_target["mask"]
optimization_target_report=optimization_target["report"]
table(list(optimization_target_report.items()),("Cible GRE de l'optimisation","Valeur"))
table([("MSE baseline / GRE mesurée",relative_image_mse(model,baseline_image)),
       ("MSE initialisation / GRE mesurée",relative_image_mse(model,initial_image))])
fig,axes=plt.subplots(1,3,figsize=(15,4),layout="constrained")
target_vmax=max(float(optimization_reference_image.max()),float(np.abs(baseline_image).max()))
for ax,image,title in zip(axes,[optimization_reference_image,baseline_image,initial_image],
                         ["Cible : GRE mesurée (échelle fixe)","Simulation EPI sans ACDC","Simulation EPI initiale"]):
    image_physical(ax,np.where(optimization_reference_mask,np.abs(image),np.nan),
        X_image,Y_image,title,vmin=0,vmax=target_vmax)
show(fig)

solution=None
image_history=None
if RUN_OPTIMIZATION:
    if GPU_REQUIRED and model.device.type!="cuda":
        raise RuntimeError("L'optimisation exige CUDA mais le modèle actuel est sur CPU. "
            "Corriger l'environnement PyTorch/CUDA, redémarrer le noyau et relancer depuis le début. "
            "Modifier GPU_PROFILE seul ne déplace pas un modèle existant.")
    table([("Calcul de l'optimisation",str(model.device)),
           ("Tenseurs du modèle",str(model.B.device)),
           ("Précision",str(model.dtype)),
           ("Carte",model.runtime_info["gpu_name"]),
           ("Stockage du cache MRzero",model.runtime_info["native_baseline_cache"]),
           ("ADC par bloc",model.sample_chunk)])
    image_history=OptimizationImageHistory(optimization_reference_image,baseline_image,every=LIVE_UPDATE_EVERY,
        title="Scan current — images de l'optimisation",origin="upper")
    image_history_npz=OUTPUT_DIR/"optimization_images.npz"
    image_history_html=OUTPUT_DIR/"optimization_images.html"
    live=LiveOptimizationPlot(reference=optimization_reference_image,baseline=baseline_image,
        current_limit_a=MAX_CURRENT_A,roi_mask=optimization_reference_mask,
        title=f"06 : alignement {ALIGNMENT_ANGLE_DEG}°, miroirs X={ALIGNMENT_MIRROR_X}, Y={ALIGNMENT_MIRROR_Y}")
    def update(snapshot):
        live.set_history(snapshot["history"]["iteration"],snapshot["history"]["fval"])
        live.update(iteration=snapshot["iteration"],image=snapshot["image"],currents_a=snapshot["currents_A"],
            loss=snapshot["fval"],nrmse_percent=100*np.sqrt(snapshot["fval"]),
            max_field_hz=snapshot["max_field_Hz"],field_limit_hz=FIELD_LIMIT_HZ)
        if image_history.record(snapshot):
            image_history.save_npz(image_history_npz)
            image_history.write_html(image_history_html)
            image_history.display()
    try:
        solution=optimize_mrzero_image(model,initial_currents,total_current_limit_A=TOTAL_CURRENT_LIMIT_A,
            field_limit_Hz=FIELD_LIMIT_HZ,max_iterations=MAX_ITERATIONS,learning_rate_A=LEARNING_RATE_A,
            target_mse=TARGET_MSE,patience=PATIENCE,callback=update,callback_every=LIVE_UPDATE_EVERY,
            use_cuda_graph=False,field_guidance_weight=FIELD_GUIDANCE_WEIGHT,
            field_guidance_scale_Hz=FIELD_GUIDANCE_SCALE_HZ)
    finally:
        live.finish("Terminé" if solution is not None else "Arrêt : dernier état évalué")
    info=solution["optimization_info"]
    currents=solution["currents_A"]
    history=solution["history"]
    table(list(info.items()))
    if "field_guidance" in info:
        guide_report=info["field_guidance"]
        table([("Coût total initial",solution["initial_objective"]),("Coût total retenu",info["objective"]),
               ("B0 résiduel RMS initial [Hz]",guide_report["initial_residual_field_rms_Hz"]),
               ("B0 résiduel RMS retenu [Hz]",guide_report["final_residual_field_rms_Hz"])])
    print("Galerie des images :",image_history_html)
else:print("Optimisation désactivée ; les diagnostics du modèle restent disponibles.")


## 7. Vérifier les courants physiques et la conversion DAC

On recalcule les limites entre les ADC, sur le TR complet, puis après conversion
nominale DAC. La MSE quantifiée doit aussi rester meilleure que celle du point initial.
En mode guidé, le coût total après DAC doit également être au plus celui du point
initial. Le poids et l’échelle utilisés sont ceux du résultat enregistré, pas une
cellule de paramètres éventuellement changée après le calcul.
Une limite d'itérations atteinte est rapportée comme telle, sans prétendre à une
convergence mathématique.

In [ ]:
solver_ok=False
if solution is not None:
    hardware_report=model.hardware.diagnostics(currents,TOTAL_CURRENT_LIMIT_A,FIELD_LIMIT_HZ)
    quantized_currents=quantize_currents(currents)
    quantized_hardware=model.hardware.diagnostics(quantized_currents,TOTAL_CURRENT_LIMIT_A,FIELD_LIMIT_HZ)
    quantized_image,quantized_kdata=model.forward(quantized_currents)
    quantized_mse=relative_image_mse(model,quantized_image)
    quantized_objective=quantized_mse
    quantized_field_guidance=None
    if info.get("field_guidance_weight",0.)>0:
        dac_guide=FieldGuidance(model,scale_Hz=info["field_guidance_scale_Hz"])
        dac_penalty,_,dac_rms=dac_guide.value_gradient(model._check_currents(quantized_currents),gradient=False)
        quantized_objective+=info["field_guidance_weight"]*float(dac_penalty)
        quantized_field_guidance={"field_penalty":float(dac_penalty),"residual_field_rms_Hz":float(dac_rms)}
    solver_ok=bool(hardware_report["all_limits_met"] and quantized_hardware["all_limits_met"]
        and np.isfinite(info["fval"]) and info["fval"]<=solution["initial_relative_mse"]+1e-7
        and np.isfinite(quantized_mse) and quantized_mse<=solution["initial_relative_mse"]+1e-7
        and np.isfinite(quantized_objective) and quantized_objective<=solution["initial_objective"]+1e-7)
    table([(key,value,quantized_hardware["checks"][key]) for key,value in hardware_report["checks"].items()],
          ("Contrôle matériel","Optimisé","Après DAC"))
    table([("MSE point initial",solution["initial_relative_mse"]),("MSE optimisée",info["fval"]),
           ("MSE après DAC",quantized_mse),("Contrôles numériques d'export",solver_ok)])
    if quantized_field_guidance is not None:
        table([("Coût total après DAC",quantized_objective),
               ("B0 résiduel RMS après DAC [Hz]",quantized_field_guidance["residual_field_rms_Hz"])])
    fig,axes=plt.subplots(1,4,figsize=(18,4),layout="constrained")
    vmax=max(abs(image).max() for image in (optimization_reference_image,baseline_image,solution["image"],quantized_image))
    for ax,image,title in zip(axes,[optimization_reference_image,baseline_image,solution["image"],quantized_image],
                             ["Cible GRE mesurée","Sans ACDC","ACDC optimisé","Après DAC"]):
        image_physical(ax,abs(image),X_image,Y_image,title,vmin=0,vmax=vmax)
    show(fig)
    fig,axes=plt.subplots(1,3,figsize=(15,4),layout="constrained")
    im=axes[0].imshow(currents,cmap="RdBu_r",aspect="auto",vmin=-MAX_CURRENT_A,vmax=MAX_CURRENT_A)
    fig.colorbar(im,ax=axes[0]);axes[0].set(title="Commandes DAC",xlabel="Ligne chronologique",ylabel="Canal")
    axes[1].plot(history["iteration"],history["fval"],label="Évalué")
    axes[1].plot(history["iteration"],history["best_fval"],label="Minimum image rencontré")
    axes[1].set(title="MSE relative",xlabel="Itération");axes[1].legend()
    axes[2].bar(np.arange(model.nc)+1,hardware_report["avg_abs_current_A"])
    axes[2].set(title="Moyenne du courant physique sur le TR",xlabel="Canal",ylabel="Moyenne |I| [A]")
    show(fig)
    if "objective" in history:
        fig,axes=plt.subplots(1,2,figsize=(11,4),layout="constrained")
        axes[0].plot(history["iteration"],history["objective"],label="Évalué")
        axes[0].plot(history["iteration"],history["best_objective"],label="Meilleur coût total")
        axes[0].set(title="Image + guidage B0",xlabel="Itération");axes[0].legend()
        axes[1].plot(history["iteration"],history["residual_field_rms_Hz"])
        axes[1].set(title="Champ résiduel physique",xlabel="Itération",ylabel="RMS [Hz]")
        show(fig)


## 8. Réponse temporelle et champs des sous-spins

La comparaison instantanée/filtrée utilise exactement les mêmes commandes. Les cartes
parentales ci-dessous sont des moyennes pour l'affichage ; les phases et les limites
ont été calculées sur chaque enfant.

In [ ]:
if solution is not None:
    response=solution["response_diagnostics"]
    instantaneous_image,_=model.forward(currents,response_enabled=False)
    fig,axes=plt.subplots(1,3,figsize=(15,4),layout="constrained")
    for ax,image,title in zip(axes,[instantaneous_image,solution["image"],abs(instantaneous_image)-abs(solution["image"])],
                             ["Même commande : réponse instantanée","Réponse filtrée 133 µs","Différence de magnitude"]):
        image_physical(ax,abs(image) if "Différence" not in title else image,X_image,Y_image,title,
                       cmap="RdBu_r" if "Différence" in title else "gray")
    show(fig)
    field_parent=response["generated_field_at_line_center_Hz"]
    fig,axes=plt.subplots(1,3,figsize=(15,4),layout="constrained")
    for ax,index in zip(axes,[0,model.nstates//2,model.nstates-1]):
        im=image_physical(ax,field_parent[...,index],prepared["physical_X_m"],prepared["physical_Y_m"],
                          f"Champ ACDC moyen : ligne {index+1} [Hz]",cmap="RdBu_r")
        fig.colorbar(im,ax=ax,shrink=.8)
    show(fig)

## 9. Sauvegarde complète de l'optimisation et des orientations

Les données enregistrées contiennent les matrices Q/P/O, les miroirs, les positions
patient des sous-spins, l'amplitude ancrée, le B0, les taux, les courants et les contrôles.
Les noms de session évitent d'écraser une optimisation précédente.
Les images périodiques sont dans une archive séparée ; seuls les courants initiaux et finaux sont sauvegardés.


In [ ]:
saved_guidance_configuration=(
    {"weight":solution["optimization_info"]["field_guidance_weight"],
     "scale_Hz":solution["optimization_info"]["field_guidance_scale_Hz"]}
    if solution is not None else
    {"weight":FIELD_GUIDANCE_WEIGHT,"scale_Hz":FIELD_GUIDANCE_SCALE_HZ})
OUTPUT_DIR.mkdir(parents=True,exist_ok=True)
arrays={"echo_times_s":inputs["echo_times_s"],
    "object_support":fit["object_support"],"noise_mask":fit["noise_mask"],
    "noise_covariance":fit["noise_covariance"],
    "support_snr":fit["support_snr"],"support_debiased_amplitude":fit["support_debiased_amplitude"],
    "fit_valid_mask":fit["fit_valid_mask"],"target_reliable_mask":fit["target_reliable_mask"],
    "R2star_sInv":fit["R2star_sInv"],"R2star_std_sInv":fit["R2star_std_sInv"],
    "ideal_image":ideal_image,"baseline_image":baseline_image,"measured_image":measured_image,
    "initial_currents_A":initial_currents,"X_image_m":X_image,"Y_image_m":Y_image,
    "field_guidance_weight":np.asarray(saved_guidance_configuration["weight"]),
    "field_guidance_scale_Hz":np.asarray(saved_guidance_configuration["scale_Hz"]),
    "channel_max_current_A":CHANNEL_MAX_CURRENT_A,
    "channel_map":np.asarray(WAVEFORM_PARAMETERS["channel_map"],dtype=int),
    "disabled_physical_channels":np.asarray(WAVEFORM_PARAMETERS["disabled_channels"],dtype=int),
    "alignment_matrix":Q,"passive_frame_matrix":P,"combined_frame_matrix":O,
    "native_X_m":prepared["native_X_m"],"native_Y_m":prepared["native_Y_m"],
    "native_amplitude":prepared["native_amplitude"],"native_rate_sInv":prepared["native_rate_sInv"],
    "native_B0_Hz":prepared["native_B0_Hz"],"parent_valid_mask":prepared["parent_valid_mask"],
    "parent_flat_index":prepared["parent_flat_index"],"subspin_weights":prepared["weights"],
    "subspin_xyz_patient_m":prepared["xyz_m"],"subspin_B0_Hz":prepared["subspin_b0"],
    "subspin_basis_Hz_per_A":prepared["fieldbasis"],"subspin_amplitude_at_reference":experiment["m0_proxy"],
    "kx_mInv":experiment["acquisition"]["kx_mInv"],"ky_mInv":experiment["acquisition"]["ky_mInv"],
    "ADC_time_s":experiment["acquisition"]["time_s"]}
arrays.update(optimization_reference_image=optimization_reference_image,
    optimization_reference_mask=optimization_reference_mask,
    optimization_reference_measured_magnitude=optimization_target["measured_reference_magnitude"],
    optimization_reference_coverage_mask=optimization_target["coverage_mask"],
    optimization_fixed_gain=np.asarray(optimization_target["gain"]),
    measured_GRE_native_magnitude=inputs["gre"]["magnitude"],
    measured_GRE_native_X_m=inputs["gre"]["X_m"],measured_GRE_native_Y_m=inputs["gre"]["Y_m"],
    zero_b0_diagnostic_image=ideal_image)
arrays.update(uncalibrated_baseline_image=uncalibrated_baseline_image,
    uncalibrated_ideal_image=uncalibrated_ideal_image,uncalibrated_baseline_kdata=uncalibrated_baseline_kdata,
    original_native_X_m=prepared["original_native_X_m"],original_native_Y_m=prepared["original_native_Y_m"],
    native_measured_B0_Hz=prepared["native_measured_B0_Hz"],
    ADC_phase_offset_rad=experiment["acquisition"].get("phase_offset_rad",np.zeros_like(experiment["acquisition"]["time_s"])),
    **{"baseline_calibration_"+key:value for key,value in baseline_calibration_masks.items()})
arrays.update({"signal_disagreement_"+key:value for key,value in baseline_signal_disagreement_arrays.items()})
if point_baseline_image is not None:
    arrays["point_baseline_image"]=point_baseline_image
if effective_simulation_model=="finite_voxel":
    for name in ("edge_vectors_m","b0_gradient_Hz_per_pixel","field_gradient_Hz_per_A_per_pixel"):
        arrays["inplane_"+name]=experiment["inplane_voxel"][name]
if solution is not None:
    arrays.update(optimized_image=solution["image"],optimized_kdata=solution["kdata"],currents_A=currents,
        quantized_currents_A=quantized_currents,quantized_image=quantized_image,
        iteration=history["iteration"],relative_mse=history["fval"],best_relative_mse=history["best_fval"])
    for key in ("objective","best_objective","field_penalty","residual_field_rms_Hz"):
        if key in history:arrays["optimization_"+key]=history[key]
if complex_echo_enabled:
    ce=fit["multi_echo"]
    arrays.update(complex_echo_width_Hz=ce["unresolved_width_Hz"],
        complex_echo_dispersion_kind=ce["dispersion_kind"],
        complex_echo_residual_R2_sInv=ce["residual_R2_sInv"],
        complex_echo_envelope_at_reference=ce["amplitude_envelope_at_reference"],
        complex_echo_refined_B0_Hz=ce["refined_b0_Hz"],
        complex_echo_predicted_RSS=ce["predicted_RSS"],
        complex_echo_hole_mask=complex_echo_diagnostics["hole_mask"],
        complex_echo_known_gradient_Hz_per_pixel=ce["known_inplane_gradient_Hz_per_pixel"],
        complex_echo_support_mask=ce["support_mask"],complex_echo_quality_mask=ce["quality_mask"],
        complex_echo_reduced_chi2=ce["reduced_chi2"],
        complex_echo_training_relative_RMSE=ce["training_relative_complex_RMSE"],
        complex_echo_validation_relative_RMSE=ce["validation_relative_complex_RMSE"],
        complex_echo_receiver_amplitude=ce["complex_receiver_amplitude"])
if complex_echo_enabled:
    for key in ("reference_amplitude_ratio","reference_amplitude_coverage_mask",
                "reference_contrast_field","reference_contrast_coverage_mask"):
        if key in fit["multi_echo"]:
            arrays[key]=fit["multi_echo"][key]
np.savez_compressed(OUTPUT_DIR/"mrzero06_optimization.npz",**arrays)
source_hashes=dict(inputs["source_sha256"],calibration=preparation_report["calibration_source_sha256"])
summary={"notebook":"06_acdc_echo32_b0_s0_t2star_MRzero.ipynb",
    "sequence_assessment":sequence_assessment,
    "input_selection":{name:str(path.resolve()) for name,path in inputs["paths"].items()},
    "reference_sequence_validation":inputs.get("reference_sequence_validation"),
    "reference_geometry_checks":inputs.get("geometry_checks"),
    "reference_amplitude_prior":fit["report"].get("reference_amplitude_prior"),
    "optimization_target":optimization_target_report,
    "field_guidance_configuration":saved_guidance_configuration,
    "zero_b0_image_role":"Diagnostic only; optimization target is the selected measured GRE.",
    "multi_echo_fit":fit["report"].get("multi_echo"),
    "multi_echo_comparison":None if complex_echo_diagnostics is None else complex_echo_diagnostics["rows"],
    "echo_input_configuration":{"B0_sequence":B0_SEQUENCE_NAME,"B0_scan":inputs["paths"]["b0"],
        "field_echo_count":inputs["field_echo_count"],"rate_mode":effective_rate_mode,"echo_signal_model":ECHO_SIGNAL_MODEL,
        "object_amplitude_source":OBJECT_AMPLITUDE_SOURCE,
        "object_contrast_sigma_mm":OBJECT_CONTRAST_SIGMA_MM,
        "optimization_reference_threshold":OPTIMIZATION_REFERENCE_THRESHOLD,
        "reference_prior_min_validation_improvement":REFERENCE_PRIOR_MIN_VALIDATION_IMPROVEMENT,
        "object_threshold_fraction":OBJECT_THRESHOLD_FRACTION,"object_min_snr":OBJECT_MIN_SNR,
        "background_corner_m":BACKGROUND_CORNER_M,"require_noise_calibration":REQUIRE_NOISE_CALIBRATION},
    "input_dir":INPUT_DIR,"calibration_path":CALIBRATION_PATH.resolve(),
    "waveform_output_dir":WAVEFORM_OUTPUT_DIR.resolve(),"baseline_scan":inputs["paths"]["scan"].name,"orientation":angles,
    "alignment_matrix":Q,"passive_frame_matrix":P,"combined_frame_matrix":O,
    "preparation":preparation_report,"echo_fit":fit["report"],"runtime":model.runtime_info,
    "fidelity_configuration":{"requested_model":SIMULATION_MODEL,"effective_model":effective_simulation_model,
        "requested_B0_weighting":B0_WEIGHTING,"effective_B0_weighting":effective_b0_weighting},
    "baseline_signal_disagreement":baseline_signal_disagreement,
    "baseline_calibration":baseline_calibration_report,"baseline_metadata_audit":baseline_metadata_audit,
    "uncalibrated_baseline_measured_comparison":uncalibrated_baseline_measured_metrics,
    "uncalibrated_baseline_vs_zeroB0_mse":uncalibrated_baseline_model_mse,
    "fidelity_comparison_stage":"before baseline calibration",
    "fidelity_comparison":fidelity_comparison,"point_baseline_measured_comparison":point_baseline_metrics,
    "zero_current_validation":reference_validation_report,
    "zero_current_reference":reference_validation_report,
    "zero_current_V4_parity":zero_current_parity,"baseline_measured_comparison":baseline_measured_metrics,
    "baseline_vs_zeroB0_mse":baseline_model_mse,"current_limits_A":{"per_channel":CHANNEL_MAX_CURRENT_A,"total":TOTAL_CURRENT_LIMIT_A},
    "current_response":CURRENT_RESPONSE,"hardware_configuration":HARDWARE,
    "disabled_channels":WAVEFORM_PARAMETERS["disabled_channels"],
    "waveform_parameters":WAVEFORM_PARAMETERS,
    "source_sha256":source_hashes,"export_checks_passed":solver_ok,
    "image_history":{"every":LIVE_UPDATE_EVERY,
        "iterations":image_history.arrays()["iterations"] if image_history is not None else [],
        "archive":"optimization_images.npz" if image_history is not None else None,
        "viewer":"optimization_images.html" if image_history is not None else None}}
summary["waveform_adc_export"]=globals().get("adc_waveform_export")
if solution is not None:
    summary.update(optimization=info,hardware=hardware_report,hardware_after_DAC=quantized_hardware,
        quantized_relative_mse=quantized_mse,initial_relative_mse=solution["initial_relative_mse"],
        initial_objective=solution["initial_objective"],quantized_objective=quantized_objective,
        quantized_field_guidance=quantized_field_guidance)
(OUTPUT_DIR/"mrzero06_optimization.json").write_text(json.dumps(json_safe(summary),indent=2,ensure_ascii=False,allow_nan=False),encoding="utf-8")
print("Résultats enregistrés :",OUTPUT_DIR)

### 9 bis. Waveforms de shimming statique et dynamique

Les deux ajustements de champ suivent les méthodes du dépôt
[ACDC_Optim](https://github.com/bughht/ACDC_Optim). Ils utilisent la même
calibration, le même repère, les contraintes matérielles et la synchronisation
du CSV image. Ils sont calculés même si `RUN_OPTIMIZATION=False` ou
`IMAGE_INITIALIZATION="zero"`. Les fichiers sont conservés dans
`INPUT_DIR/waveform`, avec leurs paramètres et contrôles.


In [ ]:
field_shim_exports=None
if EXPORT_FIELD_SHIM_WAVEFORMS:
    field_shim_exports=export_field_shim_waveforms(
        model,WAVEFORM_OUTPUT_DIR,WAVEFORM_PARAMETERS,
        total_current_limit_A=TOTAL_CURRENT_LIMIT_A,field_limit_Hz=FIELD_LIMIT_HZ,
        timing=inputs["timing"],sequence_path=inputs["paths"]["scan_sequence"],
        csv_time_origin=CSV_TIME_ORIGIN,trigger_channel=TRIGGER_CHANNEL,
        raster_s=FIELD_SHIM_RASTER_S,max_iterations=FIELD_SHIM_MAX_ITERATIONS,
        provenance=globals().get("source_hashes",{}))
    summary["field_shim_waveforms"]=field_shim_exports
    table([(name,entry["csv_file"]) for name,entry in field_shim_exports["exports"].items()],
          ("Méthode de shimming","CSV enregistré"))
else:
    print("Exports statique et dynamique désactivés.")


## 10. Export CSV facultatif, synchronisé à l'origine choisie

Le trigger ext1 du premier bloc d'imagerie est vérifié dans le fichier `.seq`.
En mode `first_imaging_trigger`, le temps du trigger est soustrait des temps RF
utilisés par la simulation. Le CSV conserve les temps en secondes et un retour à zéro.
Il contient des ampères, avec la correspondance canal/polarité choisie au début.
Le notebook produit un fichier local ; il ne programme pas le contrôleur.

Les trois méthodes exportent dans `INPUT_DIR/waveform` : `waveformN.csv` pour
l’image, `waveform_static.csv` et `waveform_dynamic.csv` pour les shims de champ.
Une relance ajoute un numéro et conserve les fichiers précédents. Les résultats
scientifiques et les figures restent dans `OUTPUT_DIR`.


Pour l’optimisation image, les états initiaux dont tous les courants sont nuls
sont retirés, puis le temps de la première commande non nulle est soustrait
aux temps restants : cette première commande est à `t=0`, sans ajout de ligne.
Les intervalles entre commandes et le retour final à zéro sont conservés.
Le décalage par rapport à l’origine vérifiée est enregistré dans les métadonnées
et utilisé pour superposer le CSV au modèle.

Chaque CSV de l’optimisation image possède aussi un compagnon
`waveformN_zero_time.csv` : mêmes lignes, mêmes courants et même ordre,
avec uniquement la première colonne remplacée par des zéros.


In [ ]:
trigger_report=sequence_trigger_timing(inputs["paths"]["scan_sequence"],inputs["timing"],channel=TRIGGER_CHANNEL)
csv_edges,csv_timing=waveform_edges_for_origin(model,origin=CSV_TIME_ORIGIN,
    trigger_rf_s=trigger_report["trigger_rf_s"],min_update_interval_s=8e-6)
summary["trigger"]=trigger_report
summary["CSV_timing"]=csv_timing
print("Origine avant recalage :",CSV_TIME_ORIGIN,"| bornes [ms] :",csv_edges[[0,-1]]*1000)
csv_file=None
csv_zero_time_file=None
if EXPORT_WAVEFORM and solution is not None and solver_ok:
    export=export_waveform(currents,csv_edges,WAVEFORM_PARAMETERS,WAVEFORM_OUTPUT_DIR,
        include_header=True,include_zero_preamble=True,include_final_zero_row=True,
        zero_time_column=False,numbered_filename=True,start_at_first_nonzero=True)
    csv_file=Path(export["csv_file"])
    summary["waveform_export"]=export
    print("Recalage à la première commande non nulle [ms] :",export["time_origin_shift_s"]*1000,
          "| états initiaux nuls retirés :",export["leading_zero_states"])
    zero_time_export=export_zero_time_copy(csv_file)
    csv_zero_time_file=Path(zero_time_export["csv_file"])
    summary["waveform_zero_time_export"]=zero_time_export
    print("CSV avec colonne temps à zéro :",csv_zero_time_file)
    print("CSV :",csv_file)
else:print("Export CSV désactivé, ou contrôles numériques non satisfaits.")
(OUTPUT_DIR/"mrzero06_optimization.json").write_text(json.dumps(json_safe(summary),indent=2,ensure_ascii=False,allow_nan=False),encoding="utf-8")

## 11. Visualiser le signal ACDC final

Les commandes DAC et les courants filtrés prédits sont tracés pour toutes les sorties,
avec un zoom sur le plus grand changement de commande. `WAVEFORM_PREVIEW_CHANNEL`
permet de choisir une sortie. Le retour à zéro et la décroissance finale sont inclus.

Les numéros de sortie, polarités et temps suivent les réglages du CSV. Si un CSV a été
créé, ses valeurs sont relues pour afficher les commandes effectivement enregistrées.
Le courant filtré est une prédiction du modèle, avec un état initial nul.

Le chemin complet du CSV exporté est affiché avant la figure.


In [ ]:
if csv_file is not None:
    print("CSV enregistré :",Path(csv_file).resolve())
else:
    print("Aucun CSV exporté pour cette session.")
    print("Dossier de sortie prévu :",WAVEFORM_OUTPUT_DIR.resolve())

waveform_report=None
if solution is not None:
    csv_origin_rf_s=trigger_report["trigger_rf_s"] if CSV_TIME_ORIGIN=="first_imaging_trigger" else 0.0
    csv_export=summary.get("waveform_export",{}) if csv_file is not None else {}
    csv_origin_rf_s+=csv_export.get("time_origin_shift_s",0.0)
    fig,waveform_report=plot_acdc_waveform(model,currents,
        origin=CSV_TIME_ORIGIN,trigger_rf_s=trigger_report["trigger_rf_s"],
        channel_map=WAVEFORM_PARAMETERS["channel_map"],
        channel_polarity=WAVEFORM_PARAMETERS["channel_polarity"],
        channel=WAVEFORM_PREVIEW_CHANNEL,csv_file=csv_file,csv_origin_rf_s=csv_origin_rf_s,
        csv_leading_zero_states=csv_export.get("leading_zero_states",0))
    fig.savefig(OUTPUT_DIR/"mrzero06_acdc_waveform.png",dpi=150)
    show(fig)
    summary["waveform_preview"]=waveform_report
    (OUTPUT_DIR/"mrzero06_optimization.json").write_text(
        json.dumps(json_safe(summary),indent=2,ensure_ascii=False,allow_nan=False),encoding="utf-8")
else:print("Lancer l'optimisation pour afficher le signal ACDC final.")


## 12. Comparer les acquisitions mesurées du dossier `results`

Si `INPUT_DIR/results` contient des scans, cette cellule affiche **GRE référence /
EPI sans ACDC / EPI corrigée** pour chacun. `CORRECTED_SCAN_NAMES=None` utilise tous les
fichiers `*scan.dat` ; une liste de noms permet de sélectionner les acquisitions.
Leurs noms sont affichés sur les figures pour identifier chaque résultat mesuré.

Comme dans le 04, un seul gain est calculé sur l'EPI **sans ACDC**, puis appliqué à
tous les scans corrigés. La normalisation GRE, le masque et l'échelle de gris sont
communs. La ROI utilise `COMPARISON_REFERENCE_THRESHOLD` fois le maximum du GRE ;
mettre `0` pour toute la couverture commune. Les images sont reconstruites par
récepteur avant RSS, puis comparées dans les coordonnées physiques natives. Les
centres de coupe, orientations et paramètres de séquence sont vérifiés.

Le GRE de référence est interpolé sur la grille EPI de la séquence sélectionnée. Rotation et miroirs s'appliquent aux
coordonnées d'affichage communes. La RMSE et la NRMSE concernent les images mesurées ;
elles restent distinctes de la MSE utilisée pour optimiser le jumeau MRzero. La
correspondance entre chaque acquisition et le CSV actuel n'est pas déduite de son nom.

Les fichiers incompatibles avec la séquence fournie ou la coupe sont signalés dans un tableau, puis exclus.

Un fichier refusé à l’ouverture est signalé et ignoré pour cette comparaison. Une fois le fichier accessible, relancer uniquement la cellule de comparaison.


In [ ]:
# Charger le correctif sans relancer les étapes de simulation.
import importlib
import measured_scan_comparison as measured_comparison_tools
compare_measured_scan_set=importlib.reload(measured_comparison_tools).compare_measured_scan_set
CORRECTED_SCAN_NAMES = ["meas_MID01843_FID49598_image_based_correct_scan.dat",
    "meas_MID01845_FID49600_static_scan.dat",
    "meas_MID01849_FID49604_image_based_correct_initzero_scan.dat"
                        ]
measured_comparison=None
results_dir=INPUT_DIR/"results"
if not results_dir.is_dir():
    print("Aucun dossier results : comparaison mesurée ignorée.")
else:
    corrected_paths=(sorted(results_dir.glob("*scan.dat")) if CORRECTED_SCAN_NAMES is None
                     else [results_dir/name for name in CORRECTED_SCAN_NAMES])
    if not corrected_paths:
        print("Aucun scan corrigé sélectionné dans",results_dir)
    else:
        measured_comparison=compare_measured_scan_set(inputs,corrected_paths,frame_matrix=O,
            normalize=NORMALIZE_COMPARISON,reference_threshold_fraction=COMPARISON_REFERENCE_THRESHOLD,
            skip_incompatible=True)
        comparison=measured_comparison
        report=comparison["report"]
        if report["skipped_scans"]:
            table([(item["filename"],item["reason"]) for item in report["skipped_scans"]],
                  ("Fichier non comparé","Motif"))
        if not comparison["images"]:
            print("Aucun scan corrigé lisible et compatible avec la séquence et la coupe de référence.")
        else:
            units=report["RMSE_units"]
            table([("Gain sans ACDC → GRE partagé",f"{comparison['gain']:.6g}"),
                   ("Échelle de normalisation GRE",f"{comparison['reference_scale']:.6g}"),
                   ("Unités RMSE",units),
                   ("Pixels du masque commun",report["common_pixels"]),
                   ("Pixels exclus faute de couverture",report["excluded_missing_coverage_pixels"])])
            table([(row["label"],f"{row['RMSE']:.6f}",f"{row['NRMSE_percent']:.3f}")
                   for row in comparison["rows"]],("Scan mesuré vs GRE","RMSE","NRMSE [%]"))
            vmax=max(float(np.nanmax(image)) for image in comparison["images"])
            baseline_row=comparison["rows"][0]
            for index,path in enumerate(map(Path,report["corrected_scan_paths"])):
                corrected_row=comparison["rows"][index+1]
                panels=[comparison["images"][0],comparison["images"][1],comparison["images"][index+2]]
                titles=["GRE référence",f"Scan sans ACDC\nRMSE = {baseline_row['RMSE']:.4f}",
                        f"Scan avec ACDC\n{path.name}\nRMSE = {corrected_row['RMSE']:.4f}"]
                fig,axes=plt.subplots(1,3,figsize=(18,5),layout="constrained",sharex=True,sharey=True)
                for ax,image,title in zip(axes,panels,titles):
                    ax.set_facecolor("black")
                    im=image_physical(ax,image,comparison["X_m"],comparison["Y_m"],title,vmin=0,vmax=vmax)
                fig.colorbar(im,ax=axes,label=f"Magnitude [{units}]")
                fig.savefig(OUTPUT_DIR/f"mrzero06_measured_comparison_{index+1}.png",dpi=150)
                show(fig)
            np.savez_compressed(OUTPUT_DIR/"measured_comparison.npz",
                images=np.stack(comparison["images"]),labels=np.asarray(comparison["labels"]),
                mask=comparison["mask"],X_m=comparison["X_m"],Y_m=comparison["Y_m"],
                gain=comparison["gain"],reference_scale=comparison["reference_scale"])
        comparison_summary=dict(report,metrics=comparison["rows"])
        (OUTPUT_DIR/"measured_comparison.json").write_text(
            json.dumps(json_safe(comparison_summary),indent=2,ensure_ascii=False,allow_nan=False),encoding="utf-8")
        summary["measured_comparison"]=comparison_summary
        (OUTPUT_DIR/"mrzero06_optimization.json").write_text(
            json.dumps(json_safe(summary),indent=2,ensure_ascii=False,allow_nan=False),encoding="utf-8")
